In [ ]:
%reset -f

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

import random
import os



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


import pandas as pd

eeg_path = "/content/drive/MyDrive/data_n_back_test/eeg/eeg.parquet"

In [ ]:
import psutil
print(psutil.virtual_memory())

svmem(total=179370471424, available=175941300224, percent=1.9, used=1921544192, free=172395610112, active=654925824, inactive=5564854272, buffers=207085568, cached=4846231552, shared=2248704, slab=319217664)


In [ ]:
df = pd.read_parquet(eeg_path)

In [ ]:
print("Original rows:", len(df))

Original rows: 15294488


In [ ]:
df = df[df["phase"] == 2].copy()

In [ ]:
print("After phase 2 filter:", len(df))
print("Subjects:", df["subject"].nunique())
print("Tests:", df["test"].unique())


After phase 2 filter: 7694147
Subjects: 16
Tests: [1 2 3]


In [ ]:
eeg_cols = [
    'EEG.AF3','EEG.F7','EEG.F3','EEG.FC5',
    'EEG.T7','EEG.P7','EEG.O1','EEG.O2',
    'EEG.P8','EEG.T8','EEG.FC6','EEG.F4',
    'EEG.F8','EEG.AF4'
]

In [ ]:
WIN = 512
STRIDE = 512   # you can switch to 512 if you want no overlap

X_windows = []
y_windows = []
subjects = []

for (subj, test), g in df.groupby(["subject", "test"], sort=False):

    if "timestamp" in df.columns and "counter" in df.columns:
        g = g.sort_values(["timestamp", "counter"], kind="mergesort")
    elif "timestamp" in df.columns:
        g = g.sort_values("timestamp", kind="mergesort")
    else:
        g = g.sort_index()

    X = g[eeg_cols].to_numpy(dtype=np.float32)
    y = g["test"].to_numpy()

    if len(X) < WIN:
        continue

    for start in range(0, len(X) - WIN + 1, STRIDE):
        end = start + WIN
        X_windows.append(X[start:end].T)
        y_windows.append(y[start])
        subjects.append(subj)

X_windows = np.array(X_windows, dtype=np.float32)
y_windows = np.array(y_windows)
subjects  = np.array(subjects)

print("X_windows:", X_windows.shape)
print("y_windows:", y_windows.shape)
print("subjects :", subjects.shape)

X_windows: (15007, 14, 512)
y_windows: (15007,)
subjects : (15007,)


In [ ]:
print("Channel order fed into model:")
for i, ch in enumerate(eeg_cols):
    print(i, ch)

tmp = g[eeg_cols].to_numpy(dtype=np.float32)
print("tmp shape:", tmp.shape)
print("first window shape before transpose:", tmp[:WIN].shape)
print("first window shape after transpose:", tmp[:WIN].T.shape)

Channel order fed into model:
0 EEG.AF3
1 EEG.F7
2 EEG.F3
3 EEG.FC5
4 EEG.T7
5 EEG.P7
6 EEG.O1
7 EEG.O2
8 EEG.P8
9 EEG.T8
10 EEG.FC6
11 EEG.F4
12 EEG.F8
13 EEG.AF4
tmp shape: (158371, 14)
first window shape before transpose: (512, 14)
first window shape after transpose: (14, 512)


In [ ]:
import numpy as np

print("Starting bandpower computation")
print("X_windows dtype:", X_windows.dtype)
print("X_windows shape:", X_windows.shape)

FS = 128
bands = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

N, C, T = X_windows.shape
print("N:", N, "C:", C, "T:", T)

# ---- FFT frequencies ----
freqs = np.fft.rfftfreq(T, d=1.0/FS)
print("freqs shape:", freqs.shape)
print("First 10 freqs:", freqs[:10])
print("Last 10 freqs:", freqs[-10:])

# ---- FFT ----
X_fft = np.fft.rfft(X_windows, axis=-1)
print("X_fft shape:", X_fft.shape)
print("X_fft dtype:", X_fft.dtype)

# ---- PSD ----
PSD = (np.abs(X_fft) ** 2) / T
print("PSD shape:", PSD.shape)
print("PSD min/max:", PSD.min(), PSD.max())

# ---- Band extraction ----
bp_list = []

for name, f_lo, f_hi in bands:
    mask = (freqs >= f_lo) & (freqs < f_hi)

    print("\nBand:", name)
    print("Freq range:", f_lo, "-", f_hi)
    print("Number of bins in mask:", mask.sum())

    bp = PSD[..., mask]
    print("PSD slice shape:", bp.shape)

    bp_mean = bp.mean(axis=-1)
    print("After mean over freq axis:", bp_mean.shape)

    bp_list.append(bp_mean)

# ---- Stack bands ----
X_bp = np.stack(bp_list, axis=-1)
print("\nFinal stacked bandpower shape:", X_bp.shape)

# ---- Flatten ----
X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)
print("Flattened shape:", X_bp_flat.shape)

print("Done bandpower.")


Starting bandpower computation
X_windows dtype: float32
X_windows shape: (15007, 14, 512)
N: 15007 C: 14 T: 512
freqs shape: (257,)
First 10 freqs: [0.   0.25 0.5  0.75 1.   1.25 1.5  1.75 2.   2.25]
Last 10 freqs: [61.75 62.   62.25 62.5  62.75 63.   63.25 63.5  63.75 64.  ]
X_fft shape: (15007, 14, 257)
X_fft dtype: complex64
PSD shape: (15007, 14, 257)
PSD min/max: 0.0 13989107000.0

Band: theta
Freq range: 4 - 8
Number of bins in mask: 16
PSD slice shape: (15007, 14, 16)
After mean over freq axis: (15007, 14)

Band: alpha
Freq range: 8 - 12
Number of bins in mask: 16
PSD slice shape: (15007, 14, 16)
After mean over freq axis: (15007, 14)

Band: beta
Freq range: 12 - 30
Number of bins in mask: 72
PSD slice shape: (15007, 14, 72)
After mean over freq axis: (15007, 14)

Final stacked bandpower shape: (15007, 14, 3)
Flattened shape: (15007, 42)
Done bandpower.


In [ ]:
# X_bp currently shape: (15007, 14, 3)

print("Before relative normalization:")
print("Example band sums (first window, first channel):",
      X_bp[0, 0].sum())

# Relative within each channel
X_bp = X_bp / (X_bp.sum(axis=-1, keepdims=True) + 1e-8)

print("After relative normalization:")
print("Example band sums (first window, first channel):",
      X_bp[0, 0].sum())

# Flatten
N = X_bp.shape[0]
X_bp_flat = X_bp.reshape(N, -1).astype(np.float32)

print("X_bp_flat shape:", X_bp_flat.shape)

Before relative normalization:
Example band sums (first window, first channel): 7497.5425
After relative normalization:
Example band sums (first window, first channel): 1.0
X_bp_flat shape: (15007, 42)


In [ ]:
import torch

X_time_t = torch.tensor(X_windows, dtype=torch.float32)
X_bp_t   = torch.tensor(X_bp_flat, dtype=torch.float32)

# Encode labels 0..2
unique_labels = np.unique(y_windows)
label_map = {v:i for i,v in enumerate(unique_labels)}
y_encoded = np.vectorize(label_map.get)(y_windows).astype(np.int64)

y_t = torch.tensor(y_encoded, dtype=torch.long)

print("X_time_t:", X_time_t.shape)
print("X_bp_t:", X_bp_t.shape)
print("y_t:", y_t.shape)
print("Label mapping:", label_map)

X_time_t: torch.Size([15007, 14, 512])
X_bp_t: torch.Size([15007, 42])
y_t: torch.Size([15007])
Label mapping: {np.int64(1): 0, np.int64(2): 1, np.int64(3): 2}


In [ ]:
# Subjects to remove
bad_subjects = ["subject_05", "subject_11", "subject_16"]

print("Original total windows:", len(subjects))

mask = ~np.isin(subjects, bad_subjects)

X_time_t = X_time_t[mask]
X_bp_t   = X_bp_t[mask]
y_t      = y_t[mask]
subjects = subjects[mask]

print("After removal total windows:", len(subjects))
print("Remaining unique subjects:", np.unique(subjects))
print("Remaining count:", len(np.unique(subjects)))

Original total windows: 15007
After removal total windows: 12159
Remaining unique subjects: ['subject_01' 'subject_02' 'subject_03' 'subject_04' 'subject_06'
 'subject_07' 'subject_08' 'subject_09' 'subject_10' 'subject_12'
 'subject_13' 'subject_14' 'subject_15']
Remaining count: 13


In [ ]:
def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    """
    proj: (B, D) projection vectors
    y:    (B,) labels
    subj: (B,) subject ids (can be ints)
    positives: same y AND different subj
    """
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature            # (B,B)
    sim = sim - sim.max(dim=1, keepdim=True).values  # stabilize

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    # exclude self
    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    # If an anchor has 0 positives in batch, we skip it (loss=0 for that anchor)
    pos_counts = pos_mask.sum(dim=1)  # (B,)
    log_prob = sim - torch.log(denom)

    # mean log-prob over positives
    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


In [ ]:

N, C, T = X_windows.shape

cov_feats = []

for i in range(N):
    X = X_windows[i]                # (14, 512)
    X = X - X.mean(axis=1, keepdims=True)  # zero-mean per channel
    cov = (X @ X.T) / (T - 1)      # (14,14)

    # take upper triangle including diagonal
    iu = np.triu_indices(C)
    cov_vec = cov[iu]              # 105 features
    cov_feats.append(cov_vec)

X_cov = np.array(cov_feats, dtype=np.float32)

print("X_cov shape:", X_cov.shape)  # should be (N, 105)

X_cov shape: (15007, 105)


In [ ]:
X_cov_t = torch.tensor(X_cov, dtype=torch.float32)

In [ ]:
all_subjects = np.unique(subjects)
print("\nSubjects list:", all_subjects)


Subjects list: ['subject_01' 'subject_02' 'subject_03' 'subject_04' 'subject_06'
 'subject_07' 'subject_08' 'subject_09' 'subject_10' 'subject_12'
 'subject_13' 'subject_14' 'subject_15']


In [ ]:
import numpy as np
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X_time, X_bp, X_cov, X_plv, X_roi, y, subj_strs):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_cov  = X_cov
        self.X_plv  = X_plv
        self.X_roi  = X_roi
        self.y      = y

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_cov[idx],
            self.X_plv[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx]
        )

In [ ]:
from torch.utils.data import Sampler
import random
from collections import defaultdict

class SubjectBalancedBatchSampler(Sampler):
    """
    Creates batches containing samples from multiple subjects.
    Works with your existing EEGDataset (which has .subj tensor).
    """

    def __init__(self, dataset, batch_size, subjects_per_batch=4):
        self.dataset = dataset
        self.batch_size = batch_size
        self.subjects_per_batch = subjects_per_batch

        # Build index list per subject
        self.subj_to_indices = defaultdict(list)
        for idx, s in enumerate(dataset.subj.tolist()):
            self.subj_to_indices[s].append(idx)

        self.subjects = list(self.subj_to_indices.keys())

        # samples per subject inside batch
        self.samples_per_subject = batch_size // subjects_per_batch

    def __iter__(self):
        # Shuffle indices per subject
        for s in self.subjects:
            random.shuffle(self.subj_to_indices[s])

        # Create subject iterators
        subj_iters = {s: iter(self.subj_to_indices[s]) for s in self.subjects}

        while True:
            # randomly choose subjects for this batch
            chosen_subjects = random.sample(self.subjects, self.subjects_per_batch)

            batch = []

            for s in chosen_subjects:
                for _ in range(self.samples_per_subject):
                    try:
                        batch.append(next(subj_iters[s]))
                    except StopIteration:
                        return  # stop when any subject runs out

            if len(batch) == self.batch_size:
                yield batch
            else:
                return

    def __len__(self):
        # approximate number of batches
        min_samples = min(len(v) for v in self.subj_to_indices.values())
        return (min_samples // self.samples_per_subject)

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt, hilbert

fs = 128
low, high = 4, 7

# Bandpass filter design
b, a = butter(4, [low/(fs/2), high/(fs/2)], btype='band')

N, C, T = X_windows.shape
plv_feats = []

for i in range(N):
    X = X_windows[i]  # (14, T)

    # Bandpass filter each channel
    X_filt = np.array([filtfilt(b, a, X[ch]) for ch in range(C)])

    # Hilbert transform → analytic signal
    analytic = hilbert(X_filt)
    phase = np.angle(analytic)  # (14, T)

    # Compute PLV matrix
    plv_mat = np.zeros((C, C))
    for ch1 in range(C):
        for ch2 in range(ch1, C):
            phase_diff = phase[ch1] - phase[ch2]
            plv = np.abs(np.mean(np.exp(1j * phase_diff)))
            plv_mat[ch1, ch2] = plv
            plv_mat[ch2, ch1] = plv

    # Upper triangle including diagonal
    iu = np.triu_indices(C)
    plv_vec = plv_mat[iu]  # 105 dims
    plv_feats.append(plv_vec)

X_plv = np.array(plv_feats, dtype=np.float32)
print("X_plv shape:", X_plv.shape)  # (N, 105)

X_plv_t = torch.tensor(X_plv, dtype=torch.float32)

X_plv shape: (15007, 105)


In [ ]:
# ----------------------------------------
# Alpha (8–12 Hz) PLV
# ----------------------------------------

low_a, high_a = 8, 12
b_a, a_a = butter(4, [low_a/(fs/2), high_a/(fs/2)], btype='band')

plv_alpha_feats = []

for i in range(N):
    X = X_windows[i]  # (14, T)

    # Bandpass filter
    X_filt = np.array([filtfilt(b_a, a_a, X[ch]) for ch in range(C)])

    analytic = hilbert(X_filt)
    phase = np.angle(analytic)

    plv_mat = np.zeros((C, C))
    for ch1 in range(C):
        for ch2 in range(ch1, C):
            phase_diff = phase[ch1] - phase[ch2]
            plv = np.abs(np.mean(np.exp(1j * phase_diff)))
            plv_mat[ch1, ch2] = plv
            plv_mat[ch2, ch1] = plv

    iu = np.triu_indices(C)
    plv_alpha_feats.append(plv_mat[iu])

X_plv_alpha = np.array(plv_alpha_feats, dtype=np.float32)
print("Alpha PLV shape:", X_plv_alpha.shape)

Alpha PLV shape: (15007, 105)


In [ ]:
# ----------------------------------------
# Combine Theta + Alpha PLV
# ----------------------------------------

X_plv_combined = np.concatenate([X_plv, X_plv_alpha], axis=1)
print("Combined PLV shape:", X_plv_combined.shape)  # (N, 210)

X_plv_t = torch.tensor(X_plv_combined, dtype=torch.float32)

Combined PLV shape: (15007, 210)


In [ ]:
def extract_embeddings(model, loader, device):

    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            E.append(e.cpu())
            Y.append(yb.cpu())

    E = torch.cat(E, dim=0)
    Y = torch.cat(Y, dim=0)

    return E, Y

In [ ]:
import torch.nn as nn

def set_partial_bn_adapt(model: nn.Module, allow=("time_branch",)):
    """
    Put BN layers in *specified submodules* into train mode (update running stats).
    Keep all other BN in eval mode.
    Always keep Dropout in eval mode.
    """
    # Default everything to eval
    model.eval()

    # Dropout always eval
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.eval()

    # Enable BN train only in allowed named submodules
    for name, module in model.named_modules():
        if any(name.startswith(a) or (f".{a}." in name) or name.endswith(a) for a in allow):
            for m in module.modules():
                if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
                    m.train()

In [ ]:
def prototype_loss(emb, y):
    """
    emb: (B, D) embedding vectors (from encoder)
    y:   (B,) class labels
    """

    B, D = emb.shape
    loss = 0.0
    count = 0

    classes = torch.unique(y)

    for c in classes:
        mask = (y == c)
        if mask.sum() < 2:
            continue

        class_emb = emb[mask]              # (Nc, D)
        proto = class_emb.mean(dim=0)      # (D,)

        loss += ((class_emb - proto)**2).sum(dim=1).mean()
        count += 1

    if count > 0:
        return loss / count
    else:
        return torch.tensor(0.0, device=emb.device)

In [ ]:
def supcon_diff_subject_loss(proj, y, subj, temperature=0.1):
    """
    proj: (B, D) projection vectors
    y:    (B,) labels
    subj: (B,) subject ids (can be ints)
    positives: same y AND different subj
    """
    proj = F.normalize(proj, dim=1)
    B = proj.size(0)

    sim = (proj @ proj.t()) / temperature            # (B,B)
    sim = sim - sim.max(dim=1, keepdim=True).values  # stabilize

    y = y.view(-1, 1)
    subj = subj.view(-1, 1)

    same_y = (y == y.t())
    diff_s = (subj != subj.t())
    pos_mask = same_y & diff_s

    # exclude self
    eye = torch.eye(B, device=proj.device, dtype=torch.bool)
    pos_mask = pos_mask & (~eye)
    neg_mask = ~eye

    exp_sim = torch.exp(sim) * neg_mask
    denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

    # If an anchor has 0 positives in batch, we skip it (loss=0 for that anchor)
    pos_counts = pos_mask.sum(dim=1)  # (B,)
    log_prob = sim - torch.log(denom)

    # mean log-prob over positives
    loss_per_i = torch.zeros(B, device=proj.device)
    valid = pos_counts > 0
    loss_per_i[valid] = -(log_prob * pos_mask.float()).sum(dim=1)[valid] / pos_counts[valid].float()

    return loss_per_i[valid].mean() if valid.any() else torch.tensor(0.0, device=proj.device)


In [ ]:
import torch

def build_roi_workload_features_from_bp(
    X_bp_flat: torch.Tensor,          # (N, 42)
    n_ch: int = 14,
    bands=("theta", "alpha", "beta"),  # must match how you built X_bp
    frontal_idx=(0, 1, 2, 3),          # CHANGE THIS to match your montage
    parietal_idx=(10, 11, 12, 13),     # CHANGE THIS to match your montage
    use_log: bool = True,
    use_relative: bool = False,        # relative bandpower within channel (theta/alpha/beta)
    eps: float = 1e-8
) -> torch.Tensor:
    """
    Returns ROI features (N, 3):
      [frontal_theta, parietal_alpha, theta_alpha_ratio]
    """
    assert X_bp_flat.ndim == 2, "X_bp_flat must be (N, 42)"
    n_bands = len(bands)
    assert X_bp_flat.shape[1] == n_ch * n_bands, "bp dim mismatch"

    # reshape -> (N, ch, band)
    X = X_bp_flat.view(-1, n_ch, n_bands)

    # optional relative power within channel
    if use_relative:
        denom = X.sum(dim=2, keepdim=True).clamp_min(eps)
        X = X / denom

    # optional log (common for EEG power)
    if use_log:
        X = torch.log(X.clamp_min(eps))

    # band indices
    band_to_i = {b: i for i, b in enumerate(bands)}
    theta_i = band_to_i["theta"]
    alpha_i = band_to_i["alpha"]

    theta_ch = X[:, :, theta_i]  # (N, ch)
    alpha_ch = X[:, :, alpha_i]  # (N, ch)

    frontal_theta = theta_ch[:, list(frontal_idx)].mean(dim=1)   # (N,)
    parietal_alpha = alpha_ch[:, list(parietal_idx)].mean(dim=1) # (N,)

    # ratio: theta_frontal / alpha_parietal
    # If using log-space, ratio becomes (log theta - log alpha) which is great + stable.
    if use_log:
        theta_alpha_ratio = frontal_theta - parietal_alpha
    else:
        theta_alpha_ratio = frontal_theta / parietal_alpha.clamp_min(eps)

    roi = torch.stack([frontal_theta, parietal_alpha, theta_alpha_ratio], dim=1)  # (N,3)
    return roi

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FusionEncoder(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=3):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim   # 42 + 3 = 45

        # ------------------
        # TIME BRANCH
        # ------------------
        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        # ------------------
        # BANDPOWER + ROI BRANCH
        # ------------------
        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),   # was 42 → now 45
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        # ------------------
        # COVARIANCE BRANCH
        # ------------------
        self.cov_branch = nn.Sequential(
            nn.Linear(105, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # PLV BRANCH
        # ------------------
        self.plv_branch = nn.Sequential(
            nn.Linear(210, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # FUSION
        # ------------------
        self.embed = nn.Sequential(
            nn.Linear(32 + 32 + 32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):

        # ------------------
        # Time branch
        # ------------------
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)  # (B,32)

        # ------------------
        # Bandpower + ROI
        # ------------------
        bp_in = torch.cat([x_bp, x_roi], dim=1)  # (B,45)
        z_bp = self.bp_branch(bp_in)

        # ------------------
        # Covariance
        # ------------------
        z_cov = self.cov_branch(x_cov)

        # ------------------
        # PLV
        # ------------------
        z_plv = self.plv_branch(x_plv)

        # ------------------
        # Fusion
        # ------------------
        z = torch.cat([z_time, z_bp, z_cov, z_plv], dim=1)
        e = self.embed(z)

        return e

In [ ]:
import numpy as np
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X_time, X_bp, X_cov, X_plv, X_roi, y, subj_strs):
        self.X_time = X_time
        self.X_bp   = X_bp
        self.X_cov  = X_cov
        self.X_plv  = X_plv
        self.X_roi  = X_roi
        self.y      = y

        uniq = np.unique(subj_strs)
        self.subj_map = {s: i for i, s in enumerate(uniq)}
        self.subj = torch.tensor([self.subj_map[s] for s in subj_strs], dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_time[idx],
            self.X_bp[idx],
            self.X_cov[idx],
            self.X_plv[idx],
            self.X_roi[idx],
            self.y[idx],
            self.subj[idx]
        )

In [ ]:
import torch.nn as nn

class ContrastiveModel(nn.Module):
    def __init__(self,
                 emb_dim=64,
                 proj_dim=64,
                 bp_dim=42,
                 roi_dim=3):

        super().__init__()

        # Main encoder (learns embedding geometry)
        self.encoder = FusionEncoder(
            emb_dim=emb_dim,
            bp_dim=bp_dim,
            roi_dim=roi_dim
        )

        # Projection head for SupCon loss
        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi):
        # Embedding
        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)

        # Projection (used only for contrastive loss)
        p = self.proj(e)

        return e, p

In [ ]:
import torch.nn.functional as F

def prototype_ce_loss(e, y, temperature=0.1):
    """
    e: (B, D) embeddings
    y: (B,) labels
    """

    classes = torch.unique(y)
    prototypes = []

    for c in classes:
        prototypes.append(e[y == c].mean(dim=0))

    prototypes = torch.stack(prototypes)  # (C, D)

    # normalize for cosine
    e_norm = F.normalize(e, dim=1)
    proto_norm = F.normalize(prototypes, dim=1)

    logits = e_norm @ proto_norm.T  # (B, C)
    logits = logits / temperature

    # map labels to [0, C-1]
    label_map = {c.item(): i for i, c in enumerate(classes)}
    y_mapped = torch.tensor([label_map[int(lbl.item())] for lbl in y],
                            device=y.device)

    loss = F.cross_entropy(logits, y_mapped)
    return loss

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 33
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = X_cov_t[train_idx].clone()
    X_cov_test  = X_cov_t[test_idx].clone()

    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # =========================
    X_roi_train = build_roi_workload_features_from_bp(
        X_bp_train,
        frontal_idx=(0,1,2,3),
        parietal_idx=(10,11,12,13),
        use_log=True,
        use_relative=False
    )
    X_roi_test = build_roi_workload_features_from_bp(
        X_bp_test,
        frontal_idx=(0,1,2,3),
        parietal_idx=(10,11,12,13),
        use_log=True,
        use_relative=False
    )

    roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    X_roi_train = (X_roi_train - roi_mu) / roi_sd
    X_roi_test  = (X_roi_test  - roi_mu) / roi_sd

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=3).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6488 | best 0.6488
Epoch 02 | zero-shot acc 0.6656 | best 0.6656
Epoch 03 | zero-shot acc 0.7035 | best 0.7035
Epoch 04 | zero-shot acc 0.7445 | best 0.7445
Epoch 05 | zero-shot acc 0.7182 | best 0.7445
Epoch 06 | zero-shot acc 0.7729 | best 0.7729
Epoch 07 | zero-shot acc 0.7340 | best 0.7729
Epoch 08 | zero-shot acc 0.7413 | best 0.7729
Epoch 09 | zero-shot acc 0.7350 | best 0.7729
Epoch 10 | zero-shot acc 0.7256 | best 0.7729
AdaBN + Mahalanobis few-shot acc: 0.816901445388794

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4239 | best 0.4239
Epoch 02 | zero-shot acc 0.5389 | best 0.5389
Epoch 03 | zero-shot acc 0.5527 | best 0.5527
Epoch 04 | zero-shot acc 0.5793 | best 0.5793
Epoch 05 | zero-shot acc 0.5346 | best 0.5793
Epoch 06 | zero-shot acc 0.5229 | best 0.5793
Epoch 07 | zero-shot acc 0.5570 | best 0.5793
Epoch 08 | zero-shot acc 0.6092 | best 0.6092
Epoch 09 | zero-shot acc 0.6187 | best 0.6

In [ ]:
'''
Change 1 — Normalize embeddings after extraction

Replace this section:

e_support = all_e[mask].to(device)
y_support = all_y[mask].to(device)
e_query   = all_e[~mask].to(device)
y_query   = all_y[~mask].to(device)

with this:

e_support = all_e[mask].to(device)
y_support = all_y[mask].to(device)
e_query   = all_e[~mask].to(device)
y_query   = all_y[~mask].to(device)

# STEP 1: normalize embeddings
e_support = F.normalize(e_support, dim=1)
e_query   = F.normalize(e_query, dim=1)
'''

'\nChange 1 — Normalize embeddings after extraction\n\nReplace this section:\n\ne_support = all_e[mask].to(device)\ny_support = all_y[mask].to(device)\ne_query   = all_e[~mask].to(device)\ny_query   = all_y[~mask].to(device)\n\nwith this:\n\ne_support = all_e[mask].to(device)\ny_support = all_y[mask].to(device)\ne_query   = all_e[~mask].to(device)\ny_query   = all_y[~mask].to(device)\n\n# STEP 1: normalize embeddings\ne_support = F.normalize(e_support, dim=1)\ne_query   = F.normalize(e_query, dim=1)\n'

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 33
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = X_cov_t[train_idx].clone()
    X_cov_test  = X_cov_t[test_idx].clone()

    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # =========================
    X_roi_train = build_roi_workload_features_from_bp(
        X_bp_train,
        frontal_idx=(0,1,2,3),
        parietal_idx=(10,11,12,13),
        use_log=True,
        use_relative=False
    )
    X_roi_test = build_roi_workload_features_from_bp(
        X_bp_test,
        frontal_idx=(0,1,2,3),
        parietal_idx=(10,11,12,13),
        use_log=True,
        use_relative=False
    )

    roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    X_roi_train = (X_roi_train - roi_mu) / roi_sd
    X_roi_test  = (X_roi_test  - roi_mu) / roi_sd

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=3).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6130 | best 0.6130
Epoch 02 | zero-shot acc 0.7087 | best 0.7087
Epoch 03 | zero-shot acc 0.6803 | best 0.7087
Epoch 04 | zero-shot acc 0.7014 | best 0.7087
Epoch 05 | zero-shot acc 0.7234 | best 0.7234
Epoch 06 | zero-shot acc 0.7192 | best 0.7234
Epoch 07 | zero-shot acc 0.7560 | best 0.7560
Epoch 08 | zero-shot acc 0.7266 | best 0.7560
Epoch 09 | zero-shot acc 0.7413 | best 0.7560
Epoch 10 | zero-shot acc 0.7108 | best 0.7560
AdaBN + Mahalanobis few-shot acc: 0.8333333730697632

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4665 | best 0.4665
Epoch 02 | zero-shot acc 0.5165 | best 0.5165
Epoch 03 | zero-shot acc 0.5080 | best 0.5165
Epoch 04 | zero-shot acc 0.5985 | best 0.5985
Epoch 05 | zero-shot acc 0.5463 | best 0.5985
Epoch 06 | zero-shot acc 0.5453 | best 0.5985
Epoch 07 | zero-shot acc 0.5708 | best 0.5985
Epoch 08 | zero-shot acc 0.5911 | best 0.5985
Epoch 09 | zero-shot acc 0.6294 | best 0.

In [ ]:
'''
Change 2 — Normalize prototypes

Replace this line:

protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])

with this:

protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])

# normalize prototype vectors
protos = F.normalize(protos, dim=1)
'''

'\nChange 2 — Normalize prototypes\n\nReplace this line:\n\nprotos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])\n\nwith this:\n\nprotos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])\n\n# normalize prototype vectors\nprotos = F.normalize(protos, dim=1)\n'

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# ---------------------------
# LOAD
# ---------------------------
heat_eeg_path = "/content/drive/MyDrive/hc_eeg.parquet"
df_heat = pd.read_parquet(heat_eeg_path)

print("Subjects:", df_heat['subject'].unique())
print("Test:", df_heat['test'].unique())
print("Phase:", df_heat['phase'].unique())

df_heat.groupby(['test', 'phase']).size()
df_heat.groupby(['subject', 'test']).size()

Subjects: ['subject_01' 'subject_02' 'subject_17' 'subject_18' 'subject_19'
 'subject_03' 'subject_06' 'subject_20' 'subject_08' 'subject_21'
 'subject_12' 'subject_22' 'subject_23' 'subject_24' 'subject_25'
 'subject_16' 'subject_26']
Test: [2 1]
Phase: [1 2]


subject     test
subject_01  1       101945
            2        98360
subject_02  1        98359
            2        97719
subject_03  1        96822
            2        95927
subject_06  1        96694
            2        96565
subject_08  1        76518
            2        97463
subject_12  1        95925
            2        97335
subject_16  1        97719
            2        99512
subject_17  1        94004
            2        82350
subject_18  1        97591
            2        97335
subject_19  1        97207
            2        97333
subject_20  1        95158
            2        98488
subject_21  1        96567
            2        97336
subject_22  1        97719
            2        98743
subject_23  1        99896
            2        99896
subject_24  1        98359
            2        95798
subject_25  1        97335
            2        98360
subject_26  1       100664
            2        97846
dtype: int64

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 33
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = X_cov_t[train_idx].clone()
    X_cov_test  = X_cov_t[test_idx].clone()

    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # =========================
    X_roi_train = build_roi_workload_features_from_bp(
        X_bp_train,
        frontal_idx=(0,1,2,3),
        parietal_idx=(10,11,12,13),
        use_log=True,
        use_relative=False
    )
    X_roi_test = build_roi_workload_features_from_bp(
        X_bp_test,
        frontal_idx=(0,1,2,3),
        parietal_idx=(10,11,12,13),
        use_log=True,
        use_relative=False
    )

    roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    X_roi_train = (X_roi_train - roi_mu) / roi_sd
    X_roi_test  = (X_roi_test  - roi_mu) / roi_sd

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=3).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6109 | best 0.6109
Epoch 02 | zero-shot acc 0.6835 | best 0.6835
Epoch 03 | zero-shot acc 0.6772 | best 0.6835
Epoch 04 | zero-shot acc 0.6604 | best 0.6835
Epoch 05 | zero-shot acc 0.7361 | best 0.7361
Epoch 06 | zero-shot acc 0.7371 | best 0.7371
Epoch 07 | zero-shot acc 0.6909 | best 0.7371
Epoch 08 | zero-shot acc 0.7171 | best 0.7371
Epoch 09 | zero-shot acc 0.6951 | best 0.7371
Epoch 10 | zero-shot acc 0.7382 | best 0.7382
AdaBN + Mahalanobis few-shot acc: 0.829812228679657

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4271 | best 0.4271
Epoch 02 | zero-shot acc 0.5250 | best 0.5250
Epoch 03 | zero-shot acc 0.4643 | best 0.5250
Epoch 04 | zero-shot acc 0.5506 | best 0.5506
Epoch 05 | zero-shot acc 0.5591 | best 0.5591
Epoch 06 | zero-shot acc 0.5708 | best 0.5708
Epoch 07 | zero-shot acc 0.5485 | best 0.5708
Epoch 08 | zero-shot acc 0.5879 | best 0.5879
Epoch 09 | zero-shot acc 0.5932 | best 0.5

In [ ]:
'''here we did the correct indices for roi'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 33
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = X_cov_t[train_idx].clone()
    X_cov_test  = X_cov_t[test_idx].clone()

    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # =========================
    X_roi_train = build_roi_workload_features_from_bp(
        X_bp_train,
        # frontal_idx=(0,1,2,3),
        frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
        # parietal_idx=(10,11,12,13),
        parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
        use_log=True,
        use_relative=False
    )
    X_roi_test = build_roi_workload_features_from_bp(
        X_bp_test,
        frontal_idx=(0,1,2,3),
        # parietal_idx=(10,11,12,13),
        parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
        use_log=True,
        use_relative=False
    )

    roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    X_roi_train = (X_roi_train - roi_mu) / roi_sd
    X_roi_test  = (X_roi_test  - roi_mu) / roi_sd

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=3).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6088 | best 0.6088
Epoch 02 | zero-shot acc 0.6793 | best 0.6793
Epoch 03 | zero-shot acc 0.7024 | best 0.7024
Epoch 04 | zero-shot acc 0.6898 | best 0.7024
Epoch 05 | zero-shot acc 0.7129 | best 0.7129
Epoch 06 | zero-shot acc 0.7466 | best 0.7466
Epoch 07 | zero-shot acc 0.6982 | best 0.7466
Epoch 08 | zero-shot acc 0.6583 | best 0.7466
Epoch 09 | zero-shot acc 0.7203 | best 0.7466
Epoch 10 | zero-shot acc 0.6824 | best 0.7466
AdaBN + Mahalanobis few-shot acc: 0.8028169274330139

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4100 | best 0.4100
Epoch 02 | zero-shot acc 0.5623 | best 0.5623
Epoch 03 | zero-shot acc 0.4633 | best 0.5623
Epoch 04 | zero-shot acc 0.5548 | best 0.5623
Epoch 05 | zero-shot acc 0.5485 | best 0.5623
Epoch 06 | zero-shot acc 0.5783 | best 0.5783
Epoch 07 | zero-shot acc 0.5761 | best 0.5783
Epoch 08 | zero-shot acc 0.6028 | best 0.6028
Epoch 09 | zero-shot acc 0.5421 | best 0.

In [ ]:
''' here we remove roi features'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 33
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = X_cov_t[train_idx].clone()
    X_cov_test  = X_cov_t[test_idx].clone()

    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6257 | best 0.6257
Epoch 02 | zero-shot acc 0.6572 | best 0.6572
Epoch 03 | zero-shot acc 0.6803 | best 0.6803
Epoch 04 | zero-shot acc 0.7192 | best 0.7192
Epoch 05 | zero-shot acc 0.7350 | best 0.7350
Epoch 06 | zero-shot acc 0.7256 | best 0.7350
Epoch 07 | zero-shot acc 0.7592 | best 0.7592
Epoch 08 | zero-shot acc 0.7245 | best 0.7592
Epoch 09 | zero-shot acc 0.7203 | best 0.7592
Epoch 10 | zero-shot acc 0.7434 | best 0.7592
AdaBN + Mahalanobis few-shot acc: 0.7981220483779907

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.3994 | best 0.3994
Epoch 02 | zero-shot acc 0.5453 | best 0.5453
Epoch 03 | zero-shot acc 0.5282 | best 0.5453
Epoch 04 | zero-shot acc 0.5474 | best 0.5474
Epoch 05 | zero-shot acc 0.5634 | best 0.5634
Epoch 06 | zero-shot acc 0.5602 | best 0.5634
Epoch 07 | zero-shot acc 0.6124 | best 0.6124
Epoch 08 | zero-shot acc 0.5335 | best 0.6124
Epoch 09 | zero-shot acc 0.5825 | best 0.

In [ ]:
'''here we removed PLV'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 33
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = X_cov_t[train_idx].clone()
    X_cov_test  = X_cov_t[test_idx].clone()

    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    # X_plv_train = X_plv_t[train_idx].clone()
    # X_plv_test  = X_plv_t[test_idx].clone()
    X_plv_train = torch.zeros_like(X_plv_t[train_idx])
    X_plv_test  = torch.zeros_like(X_plv_t[test_idx])

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5647 | best 0.5647
Epoch 02 | zero-shot acc 0.6078 | best 0.6078
Epoch 03 | zero-shot acc 0.6372 | best 0.6372
Epoch 04 | zero-shot acc 0.6709 | best 0.6709
Epoch 05 | zero-shot acc 0.6845 | best 0.6845
Epoch 06 | zero-shot acc 0.6583 | best 0.6845
Epoch 07 | zero-shot acc 0.6530 | best 0.6845
Epoch 08 | zero-shot acc 0.6677 | best 0.6845
Epoch 09 | zero-shot acc 0.6656 | best 0.6845
Epoch 10 | zero-shot acc 0.6772 | best 0.6845
AdaBN + Mahalanobis few-shot acc: 0.7312206625938416

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4334 | best 0.4334
Epoch 02 | zero-shot acc 0.4750 | best 0.4750
Epoch 03 | zero-shot acc 0.4888 | best 0.4888
Epoch 04 | zero-shot acc 0.4856 | best 0.4888
Epoch 05 | zero-shot acc 0.4867 | best 0.4888
Epoch 06 | zero-shot acc 0.4984 | best 0.4984
Epoch 07 | zero-shot acc 0.5442 | best 0.5442
Epoch 08 | zero-shot acc 0.5186 | best 0.5442
Epoch 09 | zero-shot acc 0.5154 | best 0.

In [ ]:
'''removed covariance'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 33
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    # X_cov_train = X_cov_t[train_idx].clone()
    # X_cov_test  = X_cov_t[test_idx].clone()

    # cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    # cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    # X_cov_train = (X_cov_train - cov_mu) / cov_sd
    # X_cov_test  = (X_cov_test  - cov_mu) / cov_sd
    # =========================
    # COVARIANCE DISABLED
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6120 | best 0.6120
Epoch 02 | zero-shot acc 0.6530 | best 0.6530
Epoch 03 | zero-shot acc 0.6845 | best 0.6845
Epoch 04 | zero-shot acc 0.6761 | best 0.6845
Epoch 05 | zero-shot acc 0.6982 | best 0.6982
Epoch 06 | zero-shot acc 0.7298 | best 0.7298
Epoch 07 | zero-shot acc 0.6930 | best 0.7298
Epoch 08 | zero-shot acc 0.7424 | best 0.7424
Epoch 09 | zero-shot acc 0.7182 | best 0.7424
Epoch 10 | zero-shot acc 0.6845 | best 0.7424
AdaBN + Mahalanobis few-shot acc: 0.8356807827949524

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4515 | best 0.4515
Epoch 02 | zero-shot acc 0.4803 | best 0.4803
Epoch 03 | zero-shot acc 0.5122 | best 0.5122
Epoch 04 | zero-shot acc 0.4920 | best 0.5122
Epoch 05 | zero-shot acc 0.5293 | best 0.5293
Epoch 06 | zero-shot acc 0.5293 | best 0.5293
Epoch 07 | zero-shot acc 0.5122 | best 0.5293
Epoch 08 | zero-shot acc 0.5410 | best 0.5410
Epoch 09 | zero-shot acc 0.6070 | best 0.

In [ ]:
'''removed time branch'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 33
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = X_cov_t[train_idx].clone()
    X_cov_test  = X_cov_t[test_idx].clone()

    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    # X_time_train = X_time_t[train_idx].clone()
    # X_time_test  = X_time_t[test_idx].clone()
    # =========================
    # TIME BRANCH DISABLED
    # =========================
    X_time_train = torch.zeros_like(X_time_t[train_idx])
    X_time_test  = torch.zeros_like(X_time_t[test_idx])

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6204 | best 0.6204
Epoch 02 | zero-shot acc 0.6803 | best 0.6803
Epoch 03 | zero-shot acc 0.6498 | best 0.6803
Epoch 04 | zero-shot acc 0.6435 | best 0.6803
Epoch 05 | zero-shot acc 0.6604 | best 0.6803
Epoch 06 | zero-shot acc 0.6677 | best 0.6803
Epoch 07 | zero-shot acc 0.6772 | best 0.6803
Epoch 08 | zero-shot acc 0.6393 | best 0.6803
Epoch 09 | zero-shot acc 0.7003 | best 0.7003
Epoch 10 | zero-shot acc 0.6667 | best 0.7003
AdaBN + Mahalanobis few-shot acc: 0.7887324094772339

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4430 | best 0.4430
Epoch 02 | zero-shot acc 0.4558 | best 0.4558
Epoch 03 | zero-shot acc 0.4728 | best 0.4728
Epoch 04 | zero-shot acc 0.4196 | best 0.4728
Epoch 05 | zero-shot acc 0.4302 | best 0.4728
Epoch 06 | zero-shot acc 0.4185 | best 0.4728
Epoch 07 | zero-shot acc 0.4441 | best 0.4728
Epoch 08 | zero-shot acc 0.5059 | best 0.5059
Epoch 09 | zero-shot acc 0.4356 | best 0.

In [ ]:
'''here i removed bandpower'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 33
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = X_cov_t[train_idx].clone()
    X_cov_test  = X_cov_t[test_idx].clone()

    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    # X_bp_train = X_bp_t[train_idx].clone()
    # X_bp_test  = X_bp_t[test_idx].clone()
    X_bp_train = torch.zeros_like(X_bp_t[train_idx])
    X_bp_test  = torch.zeros_like(X_bp_t[test_idx])

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5174 | best 0.5174
Epoch 02 | zero-shot acc 0.6761 | best 0.6761
Epoch 03 | zero-shot acc 0.6761 | best 0.6761
Epoch 04 | zero-shot acc 0.7340 | best 0.7340
Epoch 05 | zero-shot acc 0.7655 | best 0.7655
Epoch 06 | zero-shot acc 0.7603 | best 0.7655
Epoch 07 | zero-shot acc 0.7192 | best 0.7655
Epoch 08 | zero-shot acc 0.7645 | best 0.7655
Epoch 09 | zero-shot acc 0.7487 | best 0.7655
Epoch 10 | zero-shot acc 0.7476 | best 0.7655
AdaBN + Mahalanobis few-shot acc: 0.8403756022453308

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.2471 | best 0.2471
Epoch 02 | zero-shot acc 0.4356 | best 0.4356
Epoch 03 | zero-shot acc 0.5080 | best 0.5080
Epoch 04 | zero-shot acc 0.5101 | best 0.5101
Epoch 05 | zero-shot acc 0.5378 | best 0.5378
Epoch 06 | zero-shot acc 0.5815 | best 0.5815
Epoch 07 | zero-shot acc 0.5953 | best 0.5953
Epoch 08 | zero-shot acc 0.6134 | best 0.6134
Epoch 09 | zero-shot acc 0.6422 | best 0.

In [ ]:
'''everything gone but bandpower kept'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 33
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    # X_cov_train = X_cov_t[train_idx].clone()
    # X_cov_test  = X_cov_t[test_idx].clone()
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    # X_plv_train = X_plv_t[train_idx].clone()
    # X_plv_test  = X_plv_t[test_idx].clone()
    X_plv_train = torch.zeros_like(X_plv_t[train_idx])
    X_plv_test  = torch.zeros_like(X_plv_t[test_idx])

    # X_time_train = X_time_t[train_idx].clone()
    # X_time_test  = X_time_t[test_idx].clone()
    X_time_train = torch.zeros_like(X_time_t[train_idx])
    X_time_test  = torch.zeros_like(X_time_t[test_idx])

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5510 | best 0.5510
Epoch 02 | zero-shot acc 0.5962 | best 0.5962
Epoch 03 | zero-shot acc 0.5605 | best 0.5962
Epoch 04 | zero-shot acc 0.5731 | best 0.5962
Epoch 05 | zero-shot acc 0.5626 | best 0.5962
Epoch 06 | zero-shot acc 0.5804 | best 0.5962
Epoch 07 | zero-shot acc 0.6057 | best 0.6057
Epoch 08 | zero-shot acc 0.6267 | best 0.6267
Epoch 09 | zero-shot acc 0.6099 | best 0.6267
Epoch 10 | zero-shot acc 0.6057 | best 0.6267
AdaBN + Mahalanobis few-shot acc: 0.6654929518699646

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4313 | best 0.4313
Epoch 02 | zero-shot acc 0.4899 | best 0.4899
Epoch 03 | zero-shot acc 0.3770 | best 0.4899
Epoch 04 | zero-shot acc 0.3802 | best 0.4899
Epoch 05 | zero-shot acc 0.4260 | best 0.4899
Epoch 06 | zero-shot acc 0.3621 | best 0.4899
Epoch 07 | zero-shot acc 0.3546 | best 0.4899
Epoch 08 | zero-shot acc 0.3940 | best 0.4899
Epoch 09 | zero-shot acc 0.3706 | best 0.

In [ ]:
''' an extra layer of weights to decide which features to use (w/ only roi features deleted)'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 33
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = X_cov_t[train_idx].clone()
    X_cov_test  = X_cov_t[test_idx].clone()

    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # =====================================
    # FEATURE RELIABILITY WEIGHTING
    # =====================================
    # compute variance of each embedding dimension
    var = e_support.var(dim=0)
    # reliability = inverse variance
    weights = 1.0 / (var + 1e-6)
    # normalize weights
    weights = weights / weights.mean()
    # apply weighting
    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6257 | best 0.6257
Epoch 02 | zero-shot acc 0.6719 | best 0.6719
Epoch 03 | zero-shot acc 0.6845 | best 0.6845
Epoch 04 | zero-shot acc 0.6898 | best 0.6898
Epoch 05 | zero-shot acc 0.6824 | best 0.6898
Epoch 06 | zero-shot acc 0.6972 | best 0.6972
Epoch 07 | zero-shot acc 0.7119 | best 0.7119
Epoch 08 | zero-shot acc 0.7466 | best 0.7466
Epoch 09 | zero-shot acc 0.6877 | best 0.7466
Epoch 10 | zero-shot acc 0.7308 | best 0.7466
AdaBN + Mahalanobis few-shot acc: 0.8239436745643616

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4100 | best 0.4100
Epoch 02 | zero-shot acc 0.4633 | best 0.4633
Epoch 03 | zero-shot acc 0.5538 | best 0.5538
Epoch 04 | zero-shot acc 0.5463 | best 0.5538
Epoch 05 | zero-shot acc 0.5580 | best 0.5580
Epoch 06 | zero-shot acc 0.5399 | best 0.5580
Epoch 07 | zero-shot acc 0.5996 | best 0.5996
Epoch 08 | zero-shot acc 0.5389 | best 0.5996
Epoch 09 | zero-shot acc 0.5740 | best 0.

In [ ]:
''' an extra layer of weights to decide which features to use (w/ covariance and roi features deleted)'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 33
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    # X_cov_train = X_cov_t[train_idx].clone()
    # X_cov_test  = X_cov_t[test_idx].clone()

    # cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    # cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    # X_cov_train = (X_cov_train - cov_mu) / cov_sd
    # X_cov_test  = (X_cov_test  - cov_mu) / cov_sd
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # =====================================
    # FEATURE RELIABILITY WEIGHTING
    # =====================================
    # compute variance of each embedding dimension
    var = e_support.var(dim=0)
    # reliability = inverse variance
    weights = 1.0 / (var + 1e-6)
    # normalize weights
    weights = weights / weights.mean()
    # apply weighting
    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6057 | best 0.6057
Epoch 02 | zero-shot acc 0.6341 | best 0.6341
Epoch 03 | zero-shot acc 0.6919 | best 0.6919
Epoch 04 | zero-shot acc 0.7035 | best 0.7035
Epoch 05 | zero-shot acc 0.7140 | best 0.7140
Epoch 06 | zero-shot acc 0.7003 | best 0.7140
Epoch 07 | zero-shot acc 0.7098 | best 0.7140
Epoch 08 | zero-shot acc 0.7129 | best 0.7140
Epoch 09 | zero-shot acc 0.7119 | best 0.7140
Epoch 10 | zero-shot acc 0.7087 | best 0.7140
AdaBN + Mahalanobis few-shot acc: 0.8603286743164062

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4941 | best 0.4941
Epoch 02 | zero-shot acc 0.5059 | best 0.5059
Epoch 03 | zero-shot acc 0.5612 | best 0.5612
Epoch 04 | zero-shot acc 0.5623 | best 0.5623
Epoch 05 | zero-shot acc 0.5719 | best 0.5719
Epoch 06 | zero-shot acc 0.5676 | best 0.5719
Epoch 07 | zero-shot acc 0.5453 | best 0.5719
Epoch 08 | zero-shot acc 0.5485 | best 0.5719
Epoch 09 | zero-shot acc 0.4931 | best 0.

In [ ]:
'''few shot changed to 20'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    # X_cov_train = X_cov_t[train_idx].clone()
    # X_cov_test  = X_cov_t[test_idx].clone()

    # cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    # cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    # X_cov_train = (X_cov_train - cov_mu) / cov_sd
    # X_cov_test  = (X_cov_test  - cov_mu) / cov_sd
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # =====================================
    # FEATURE RELIABILITY WEIGHTING
    # =====================================
    # compute variance of each embedding dimension
    var = e_support.var(dim=0)
    # reliability = inverse variance
    weights = 1.0 / (var + 1e-6)
    # normalize weights
    weights = weights / weights.mean()
    # apply weighting
    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6225 | best 0.6225
Epoch 02 | zero-shot acc 0.6688 | best 0.6688
Epoch 03 | zero-shot acc 0.6824 | best 0.6824
Epoch 04 | zero-shot acc 0.6961 | best 0.6961
Epoch 05 | zero-shot acc 0.7140 | best 0.7140
Epoch 06 | zero-shot acc 0.6930 | best 0.7140
Epoch 07 | zero-shot acc 0.7403 | best 0.7403
Epoch 08 | zero-shot acc 0.7171 | best 0.7403
Epoch 09 | zero-shot acc 0.7045 | best 0.7403
Epoch 10 | zero-shot acc 0.6972 | best 0.7403
AdaBN + Mahalanobis few-shot acc: 0.8597082495689392

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4654 | best 0.4654
Epoch 02 | zero-shot acc 0.5570 | best 0.5570
Epoch 03 | zero-shot acc 0.5346 | best 0.5570
Epoch 04 | zero-shot acc 0.5857 | best 0.5857
Epoch 05 | zero-shot acc 0.5016 | best 0.5857
Epoch 06 | zero-shot acc 0.5495 | best 0.5857
Epoch 07 | zero-shot acc 0.5165 | best 0.5857
Epoch 08 | zero-shot acc 0.6550 | best 0.6550
Epoch 09 | zero-shot acc 0.5240 | best 0.

In [ ]:
''' each class has its own covariance matrix'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results = []

for fold_i, test_subj in enumerate(all_subjects):

    print("\n" + "="*80)
    print(f"FOLD {fold_i+1}/{len(all_subjects)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    # X_cov_train = X_cov_t[train_idx].clone()
    # X_cov_test  = X_cov_t[test_idx].clone()

    # cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    # cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    # X_cov_train = (X_cov_train - cov_mu) / cov_sd
    # X_cov_test  = (X_cov_test  - cov_mu) / cov_sd
    X_cov_train = torch.zeros_like(X_cov_t[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t[test_idx])

    X_plv_train = X_plv_t[train_idx].clone()
    X_plv_test  = X_plv_t[test_idx].clone()

    X_time_train = X_time_t[train_idx].clone()
    X_time_test  = X_time_t[test_idx].clone()

    X_bp_train = X_bp_t[train_idx].clone()
    X_bp_test  = X_bp_t[test_idx].clone()

    y_train = y_t[train_idx].clone()
    y_test  = y_t[test_idx].clone()

    subj_train = subjects[train_idx]
    subj_test  = subjects[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI WORKLOAD FEATURES
    # # =========================
    # X_roi_train = build_roi_workload_features_from_bp(
    #     X_bp_train,
    #     # frontal_idx=(0,1,2,3),
    #     frontal_idx = (0,1,2,3),  # AF3, F7, F3, FC5
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )
    # X_roi_test = build_roi_workload_features_from_bp(
    #     X_bp_test,
    #     frontal_idx=(0,1,2,3),
    #     # parietal_idx=(10,11,12,13),
    #     parietal_idx = (5,6,7,8),   # P7, O1, O2, P8
    #     use_log=True,
    #     use_relative=False
    # )

    # roi_mu = X_roi_train.mean(dim=0, keepdim=True)
    # roi_sd = X_roi_train.std(dim=0, keepdim=True) + 1e-6
    # X_roi_train = (X_roi_train - roi_mu) / roi_sd
    # X_roi_test  = (X_roi_test  - roi_mu) / roi_sd
    # ROI disabled
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(X_time_train, X_bp_train, X_cov_train, X_plv_train, X_roi_train, y_train, subj_train)
    test_ds  = EEGDataset(X_time_test,  X_bp_test,  X_cov_test,  X_plv_test,  X_roi_test,  y_test,  subj_test)

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL (metric version: NO n_classes arg)
    # =========================
    model = ContrastiveModel(emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=0).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL (train prototypes -> test classify)
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            # cosine logits
            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            # map test labels to local indices
            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    # load best encoder before adaptation
    model.load_state_dict(best_state)

    # =====================================================
    # FEW-SHOT: AdaBN + Mahalanobis (UNCHANGED)
    # =====================================================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    # STEP 1: normalize embeddings
    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # =====================================
    # FEATURE RELIABILITY WEIGHTING
    # =====================================
    # compute variance of each embedding dimension
    var = e_support.var(dim=0)
    # reliability = inverse variance
    weights = 1.0 / (var + 1e-6)
    # normalize weights
    weights = weights / weights.mean()
    # apply weighting
    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    # normalize prototype vectors
    protos = F.normalize(protos, dim=1)

    # residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    # D = residuals.shape[1]
    # cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    # avg_var = torch.trace(cov) / D
    # cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    # cov_inv = torch.linalg.pinv(cov_shrink)

    # diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    # dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)
    cov_inv_list = []
    for i, c in enumerate(classes):
      class_samples = e_support[y_support == c]

      residuals = class_samples - protos[i]

      D = residuals.shape[1]

      cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

      avg_var = torch.trace(cov) / D
      cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))

      cov_inv = torch.linalg.pinv(cov_shrink)

      cov_inv_list.append(cov_inv)

    cov_inv = torch.stack(cov_inv_list)
    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,kdd,nkd->nk", diffs, cov_inv, diffs)


    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results.append((test_subj, fewshot_acc))


print("\nMETRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results]
for s, a in overall_results:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


FOLD 1/13 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6193 | best 0.6193
Epoch 02 | zero-shot acc 0.6456 | best 0.6456
Epoch 03 | zero-shot acc 0.6761 | best 0.6761
Epoch 04 | zero-shot acc 0.6919 | best 0.6919
Epoch 05 | zero-shot acc 0.7066 | best 0.7066
Epoch 06 | zero-shot acc 0.7003 | best 0.7066
Epoch 07 | zero-shot acc 0.7487 | best 0.7487
Epoch 08 | zero-shot acc 0.7003 | best 0.7487
Epoch 09 | zero-shot acc 0.7245 | best 0.7487
Epoch 10 | zero-shot acc 0.7550 | best 0.7550
AdaBN + Mahalanobis few-shot acc: 0.8193041682243347

FOLD 2/13 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4856 | best 0.4856
Epoch 02 | zero-shot acc 0.5346 | best 0.5346
Epoch 03 | zero-shot acc 0.5655 | best 0.5655
Epoch 04 | zero-shot acc 0.6155 | best 0.6155
Epoch 05 | zero-shot acc 0.5186 | best 0.6155
Epoch 06 | zero-shot acc 0.5793 | best 0.6155
Epoch 07 | zero-shot acc 0.5463 | best 0.6155
Epoch 08 | zero-shot acc 0.6113 | best 0.6155
Epoch 09 | zero-shot acc 0.5591 | best 0.

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# ---------------------------
# LOAD
# ---------------------------
heat_eeg_path = "/content/drive/MyDrive/hc_eeg.parquet"
df_heat = pd.read_parquet(heat_eeg_path)


In [ ]:
# =========================
# EEG CHANNELS (LOCKED ORDER)
# =========================
eeg_cols_htc = [
    'EEG.AF3','EEG.F7','EEG.F3','EEG.FC5',
    'EEG.T7','EEG.P7','EEG.O1','EEG.O2',
    'EEG.P8','EEG.T8','EEG.FC6','EEG.F4',
    'EEG.F8','EEG.AF4'
]

# =========================
# FILTER TASK PHASE
# =========================
df_htc = df_heat[df_heat["phase"] == 2].copy()

# labels: 1→0, 2→1
df_htc["label"] = df_htc["test"] - 1

print("Filtered df_htc:", df_htc.shape)

Filtered df_htc: (2597172, 130)


In [ ]:
WIN_HTC = 512
STRIDE_HTC = 512

X_windows_htc = []
y_windows_htc = []
subjects_htc = []

for (subj, test), g in df_htc.groupby(["subject", "test"], sort=False):

    g = g.sort_values(["timestamp", "EEG.Counter"], kind="mergesort")

    X = g[eeg_cols_htc].to_numpy(dtype=np.float32)
    y = g["label"].to_numpy()

    if len(X) < WIN_HTC:
        continue

    for start in range(0, len(X) - WIN_HTC + 1, STRIDE_HTC):
        end = start + WIN_HTC

        X_windows_htc.append(X[start:end].T)  # (14, 512)
        y_windows_htc.append(y[start])
        subjects_htc.append(subj)

X_windows_htc = np.array(X_windows_htc, dtype=np.float32)
y_windows_htc = np.array(y_windows_htc)
subjects_htc  = np.array(subjects_htc)

print("X_windows_htc:", X_windows_htc.shape)
print("y_windows_htc:", y_windows_htc.shape)
print("subjects_htc :", subjects_htc.shape)

X_windows_htc: (5056, 14, 512)
y_windows_htc: (5056,)
subjects_htc : (5056,)


In [ ]:
FS = 128
bands = [
    ("theta", 4, 8),
    ("alpha", 8, 12),
    ("beta", 12, 30),
]

N_htc, C_htc, T_htc = X_windows_htc.shape

freqs_htc = np.fft.rfftfreq(T_htc, d=1.0/FS)
X_fft_htc = np.fft.rfft(X_windows_htc, axis=-1)
PSD_htc = (np.abs(X_fft_htc) ** 2) / T_htc

bp_list_htc = []

for name, f_lo, f_hi in bands:
    mask = (freqs_htc >= f_lo) & (freqs_htc < f_hi)
    bp = PSD_htc[..., mask].mean(axis=-1)
    bp_list_htc.append(bp)

X_bp_htc = np.stack(bp_list_htc, axis=-1)

# relative normalization (important)
X_bp_htc = X_bp_htc / (X_bp_htc.sum(axis=-1, keepdims=True) + 1e-8)

X_bp_flat_htc = X_bp_htc.reshape(N_htc, -1).astype(np.float32)

print("X_bp_flat_htc:", X_bp_flat_htc.shape)  # (N, 42)

X_bp_flat_htc: (5056, 42)


In [ ]:
from scipy.signal import butter, filtfilt, hilbert

fs = 128
low, high = 4, 7

b, a = butter(4, [low/(fs/2), high/(fs/2)], btype='band')

plv_feats_htc = []

for i in range(N_htc):
    X = X_windows_htc[i]

    X_filt = np.array([filtfilt(b, a, X[ch]) for ch in range(C_htc)])
    analytic = hilbert(X_filt)
    phase = np.angle(analytic)

    plv_mat = np.zeros((C_htc, C_htc))
    for ch1 in range(C_htc):
        for ch2 in range(ch1, C_htc):
            diff = phase[ch1] - phase[ch2]
            plv = np.abs(np.mean(np.exp(1j * diff)))
            plv_mat[ch1, ch2] = plv
            plv_mat[ch2, ch1] = plv

    iu = np.triu_indices(C_htc)
    plv_feats_htc.append(plv_mat[iu])

X_plv_htc = np.array(plv_feats_htc, dtype=np.float32)

print("X_plv_htc:", X_plv_htc.shape)  # (N, 105)

X_plv_htc: (5056, 105)


In [ ]:
X_cov_htc = np.zeros((N_htc, 105), dtype=np.float32)
X_roi_htc = np.zeros((N_htc, 0), dtype=np.float32)

In [ ]:
X_time_t_htc = torch.tensor(X_windows_htc, dtype=torch.float32)
X_bp_t_htc   = torch.tensor(X_bp_flat_htc, dtype=torch.float32)
X_cov_t_htc  = torch.tensor(X_cov_htc, dtype=torch.float32)
X_plv_t_htc  = torch.tensor(X_plv_htc, dtype=torch.float32)
X_roi_t_htc  = torch.tensor(X_roi_htc, dtype=torch.float32)

y_t_htc = torch.tensor(y_windows_htc, dtype=torch.long)

print("X_time_t_htc:", X_time_t_htc.shape)
print("X_bp_t_htc:", X_bp_t_htc.shape)
print("X_plv_t_htc:", X_plv_t_htc.shape)
print("y_t_htc:", y_t_htc.shape)

X_time_t_htc: torch.Size([5056, 14, 512])
X_bp_t_htc: torch.Size([5056, 42])
X_plv_t_htc: torch.Size([5056, 105])
y_t_htc: torch.Size([5056])


In [ ]:
all_subjects_htc = np.unique(subjects_htc)
print("HTC subjects:", all_subjects_htc)

HTC subjects: ['subject_01' 'subject_02' 'subject_03' 'subject_06' 'subject_08'
 'subject_12' 'subject_16' 'subject_17' 'subject_18' 'subject_19'
 'subject_20' 'subject_21' 'subject_22' 'subject_23' 'subject_24'
 'subject_25' 'subject_26']


In [ ]:
from scipy.signal import butter, filtfilt, hilbert

fs = 128

# ----------------------
# THETA (4–7 Hz)
# ----------------------
b_t, a_t = butter(4, [4/(fs/2), 7/(fs/2)], btype='band')

# ----------------------
# ALPHA (8–12 Hz)
# ----------------------
b_a, a_a = butter(4, [8/(fs/2), 12/(fs/2)], btype='band')

plv_theta = []
plv_alpha = []

for i in range(N_htc):
    X = X_windows_htc[i]

    # ---------- THETA ----------
    X_filt_t = np.array([filtfilt(b_t, a_t, X[ch]) for ch in range(C_htc)])
    phase_t = np.angle(hilbert(X_filt_t))

    plv_mat_t = np.zeros((C_htc, C_htc))
    for ch1 in range(C_htc):
        for ch2 in range(ch1, C_htc):
            diff = phase_t[ch1] - phase_t[ch2]
            val = np.abs(np.mean(np.exp(1j * diff)))
            plv_mat_t[ch1, ch2] = val
            plv_mat_t[ch2, ch1] = val

    iu = np.triu_indices(C_htc)
    plv_theta.append(plv_mat_t[iu])

    # ---------- ALPHA ----------
    X_filt_a = np.array([filtfilt(b_a, a_a, X[ch]) for ch in range(C_htc)])
    phase_a = np.angle(hilbert(X_filt_a))

    plv_mat_a = np.zeros((C_htc, C_htc))
    for ch1 in range(C_htc):
        for ch2 in range(ch1, C_htc):
            diff = phase_a[ch1] - phase_a[ch2]
            val = np.abs(np.mean(np.exp(1j * diff)))
            plv_mat_a[ch1, ch2] = val
            plv_mat_a[ch2, ch1] = val

    plv_alpha.append(plv_mat_a[iu])

# combine
X_plv_htc = np.concatenate([plv_theta, plv_alpha], axis=1).astype(np.float32)

print("X_plv_htc:", X_plv_htc.shape)  # should be (N, 210)

X_plv_htc: (5056, 210)


In [ ]:
X_plv_t_htc = torch.tensor(X_plv_htc, dtype=torch.float32)

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results_htc = []

for fold_i, test_subj in enumerate(all_subjects_htc):

    print("\n" + "="*80)
    print(f"HTC FOLD {fold_i+1}/{len(all_subjects_htc)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects_htc == test_subj)
    train_mask = ~test_mask
    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_cov_train = torch.zeros_like(X_cov_t_htc[train_idx])
    X_cov_test  = torch.zeros_like(X_cov_t_htc[test_idx])

    X_plv_train = X_plv_t_htc[train_idx].clone()
    X_plv_test  = X_plv_t_htc[test_idx].clone()

    X_time_train = X_time_t_htc[train_idx].clone()
    X_time_test  = X_time_t_htc[test_idx].clone()

    X_bp_train = X_bp_t_htc[train_idx].clone()
    X_bp_test  = X_bp_t_htc[test_idx].clone()

    y_train = y_t_htc[train_idx].clone()
    y_test  = y_t_htc[test_idx].clone()

    subj_train = subjects_htc[train_idx]
    subj_test  = subjects_htc[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # =========================
    # ROI (DISABLED)
    # =========================
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train,
        X_plv_train, X_roi_train,
        y_train, subj_train
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test,
        X_plv_test, X_roi_test,
        y_test, subj_test
    )

    sampler = SubjectBalancedBatchSampler(train_ds, batch_size=BATCH, subjects_per_batch=12)
    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():
            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([e_train[y_train_full == c].mean(dim=0) for c in classes])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor([label_map[int(v.item())] for v in y_test_full], device=pred_local.device)

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT + MAHALANOBIS
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e_pre, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    for p in model.parameters():
        p.requires_grad = False

    model.eval()

    set_partial_bn_adapt(
        model,
        allow=("encoder.time_branch",
               "encoder.bp_branch",
               "encoder.cov_branch",
               "encoder.plv_branch")
    )

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in support_loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            _ = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

    model.eval()

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([e_support[y_support == c] - protos[i] for i, c in enumerate(classes)], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("HTC AdaBN + Mahalanobis few-shot acc:", fewshot_acc)
    overall_results_htc.append((test_subj, fewshot_acc))


print("\nHTC METRIC TRAIN + ADABN + MAHALANOBIS LOSO SUMMARY")
accs = [a for _, a in overall_results_htc]
for s, a in overall_results_htc:
    print(s, ":", a)
print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


HTC FOLD 1/17 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.6254 | best 0.6254
Epoch 02 | zero-shot acc 0.6189 | best 0.6254
Epoch 03 | zero-shot acc 0.5570 | best 0.6254
Epoch 04 | zero-shot acc 0.6417 | best 0.6417
Epoch 05 | zero-shot acc 0.5831 | best 0.6417
Epoch 06 | zero-shot acc 0.5603 | best 0.6417
Epoch 07 | zero-shot acc 0.6026 | best 0.6417
Epoch 08 | zero-shot acc 0.6189 | best 0.6417
Epoch 09 | zero-shot acc 0.6319 | best 0.6417
Epoch 10 | zero-shot acc 0.5928 | best 0.6417
HTC AdaBN + Mahalanobis few-shot acc: 0.7078651785850525

HTC FOLD 2/17 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.6756 | best 0.6756
Epoch 02 | zero-shot acc 0.6656 | best 0.6756
Epoch 03 | zero-shot acc 0.6488 | best 0.6756
Epoch 04 | zero-shot acc 0.6756 | best 0.6756
Epoch 05 | zero-shot acc 0.5284 | best 0.6756
Epoch 06 | zero-shot acc 0.5920 | best 0.6756
Epoch 07 | zero-shot acc 0.4816 | best 0.6756
Epoch 08 | zero-shot acc 0.5351 | best 0.6756
Epoch 09 | zero-shot acc 0.52

In [ ]:
import numpy as np
from collections import defaultdict

subj_counts = defaultdict(lambda: defaultdict(int))

for i in range(len(subjects_htc)):
    s = subjects_htc[i]
    y = int(y_windows_htc[i])
    subj_counts[s][y] += 1

# print nicely
for s in sorted(subj_counts.keys()):
    total = sum(subj_counts[s].values())
    print(f"{s}: total={total}, per-class={dict(subj_counts[s])}")

subject_01: total=307, per-class={1: 152, 0: 155}
subject_02: total=299, per-class={1: 149, 0: 150}
subject_03: total=297, per-class={1: 148, 0: 149}
subject_06: total=294, per-class={1: 147, 0: 147}
subject_08: total=282, per-class={1: 149, 0: 133}
subject_12: total=297, per-class={1: 148, 0: 149}
subject_16: total=300, per-class={1: 151, 0: 149}
subject_17: total=296, per-class={1: 148, 0: 148}
subject_18: total=296, per-class={1: 148, 0: 148}
subject_19: total=297, per-class={1: 149, 0: 148}
subject_20: total=299, per-class={1: 151, 0: 148}
subject_21: total=297, per-class={1: 149, 0: 148}
subject_22: total=300, per-class={1: 151, 0: 149}
subject_23: total=298, per-class={1: 149, 0: 149}
subject_24: total=295, per-class={1: 145, 0: 150}
subject_25: total=299, per-class={1: 151, 0: 148}
subject_26: total=303, per-class={1: 149, 0: 154}


In [ ]:
print("HTC subjects:", np.unique(subjects_htc)[:5])
print("N-back subjects:", np.unique(subjects)[:5])

HTC subjects: ['subject_01' 'subject_02' 'subject_03' 'subject_06' 'subject_08']
N-back subjects: ['subject_01' 'subject_02' 'subject_03' 'subject_04' 'subject_06']


In [ ]:
print("HTC labels:", torch.unique(y_t_htc))

HTC labels: tensor([0, 1])


In [ ]:
print(C_htc)

14


In [ ]:
print("HTC shapes check:")
print("X_time:", X_time_t_htc.shape)
print("X_bp:", X_bp_t_htc.shape)
print("X_plv:", X_plv_t_htc.shape)
print("y:", y_t_htc.shape)

print("\nSubject check:")
print("Unique subjects:", len(np.unique(subjects_htc)))

print("\nPer-subject windows:")
for s in np.unique(subjects_htc):
    print(s, np.sum(subjects_htc == s))

print("\nLabel balance:")
for s in np.unique(subjects_htc):
    mask = subjects_htc == s
    vals, counts = np.unique(y_windows_htc[mask], return_counts=True)
    print(s, dict(zip(vals, counts)))

HTC shapes check:
X_time: torch.Size([5056, 14, 512])
X_bp: torch.Size([5056, 42])
X_plv: torch.Size([5056, 210])
y: torch.Size([5056])

Subject check:
Unique subjects: 17

Per-subject windows:
subject_01 307
subject_02 299
subject_03 297
subject_06 294
subject_08 282
subject_12 297
subject_16 300
subject_17 296
subject_18 296
subject_19 297
subject_20 299
subject_21 297
subject_22 300
subject_23 298
subject_24 295
subject_25 299
subject_26 303

Label balance:
subject_01 {np.int64(0): np.int64(155), np.int64(1): np.int64(152)}
subject_02 {np.int64(0): np.int64(150), np.int64(1): np.int64(149)}
subject_03 {np.int64(0): np.int64(149), np.int64(1): np.int64(148)}
subject_06 {np.int64(0): np.int64(147), np.int64(1): np.int64(147)}
subject_08 {np.int64(0): np.int64(133), np.int64(1): np.int64(149)}
subject_12 {np.int64(0): np.int64(149), np.int64(1): np.int64(148)}
subject_16 {np.int64(0): np.int64(149), np.int64(1): np.int64(151)}
subject_17 {np.int64(0): np.int64(148), np.int64(1): np.int

In [ ]:
print("Mean bandpower (first 10 dims):")
print(X_bp_t_htc.mean(dim=0)[:10])

print("Std bandpower (first 10 dims):")
print(X_bp_t_htc.std(dim=0)[:10])

Mean bandpower (first 10 dims):
tensor([0.2191, 0.3309, 0.4501, 0.2227, 0.3181, 0.4593, 0.2142, 0.3418, 0.4440,
        0.2067])
Std bandpower (first 10 dims):
tensor([0.1098, 0.1156, 0.1374, 0.1101, 0.1213, 0.1367, 0.1129, 0.1273, 0.1480,
        0.1013])


In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 5   # ↓ reduced to avoid overfitting
BATCH  = 396
LR     = 1e-3
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results_htc = []

for fold_i, test_subj in enumerate(all_subjects_htc):

    print("\n" + "="*80)
    print(f"HTC FOLD {fold_i+1}/{len(all_subjects_htc)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects_htc == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_time_train = X_time_t_htc[train_idx].clone()
    X_time_test  = X_time_t_htc[test_idx].clone()

    X_bp_train = X_bp_t_htc[train_idx].clone()
    X_bp_test  = X_bp_t_htc[test_idx].clone()

    # 🔥 COVARIANCE ENABLED
    X_cov_train = X_cov_t_htc[train_idx].clone()
    X_cov_test  = X_cov_t_htc[test_idx].clone()

    X_plv_train = X_plv_t_htc[train_idx].clone()
    X_plv_test  = X_plv_t_htc[test_idx].clone()

    y_train = y_t_htc[train_idx].clone()
    y_test  = y_t_htc[test_idx].clone()

    subj_train = subjects_htc[train_idx]
    subj_test  = subjects_htc[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================

    # TIME
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # BANDPOWER
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # 🔥 COV NORMALIZATION (NEW)
    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    # ROI disabled (consistent with your setup)
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train,
        X_plv_train, X_roi_train, y_train, subj_train
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test,
        X_plv_test, X_roi_test, y_test, subj_test
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds, batch_size=BATCH, subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT (UNCHANGED)
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("HTC AdaBN + Mahalanobis few-shot acc:", fewshot_acc)

    overall_results_htc.append((test_subj, fewshot_acc))


print("\nHTC FINAL SUMMARY")
accs = [a for _, a in overall_results_htc]
for s, a in overall_results_htc:
    print(s, ":", a)

print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


HTC FOLD 1/17 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5212 | best 0.5212
Epoch 02 | zero-shot acc 0.5537 | best 0.5537
Epoch 03 | zero-shot acc 0.6254 | best 0.6254
Epoch 04 | zero-shot acc 0.7134 | best 0.7134
Epoch 05 | zero-shot acc 0.6938 | best 0.7134
HTC AdaBN + Mahalanobis few-shot acc: 0.7265917658805847

HTC FOLD 2/17 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.5920 | best 0.5920
Epoch 02 | zero-shot acc 0.6957 | best 0.6957
Epoch 03 | zero-shot acc 0.6522 | best 0.6957
Epoch 04 | zero-shot acc 0.5786 | best 0.6957
Epoch 05 | zero-shot acc 0.5151 | best 0.6957
HTC AdaBN + Mahalanobis few-shot acc: 0.837837815284729

HTC FOLD 3/17 | TEST SUBJECT = subject_03
Epoch 01 | zero-shot acc 0.5724 | best 0.5724
Epoch 02 | zero-shot acc 0.5690 | best 0.5724
Epoch 03 | zero-shot acc 0.5556 | best 0.5724
Epoch 04 | zero-shot acc 0.6296 | best 0.6296
Epoch 05 | zero-shot acc 0.5690 | best 0.6296
HTC AdaBN + Mahalanobis few-shot acc: 0.7782101035118103

HTC FOLD 4

In [ ]:
''' here i made learning rate 5e-4, i made it smaller from what it was before 1e-4'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10  # ↓ reduced to avoid overfitting
BATCH  = 396
LR     = 5e-4
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.4

overall_results_htc = []

for fold_i, test_subj in enumerate(all_subjects_htc):

    print("\n" + "="*80)
    print(f"HTC FOLD {fold_i+1}/{len(all_subjects_htc)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects_htc == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_time_train = X_time_t_htc[train_idx].clone()
    X_time_test  = X_time_t_htc[test_idx].clone()

    X_bp_train = X_bp_t_htc[train_idx].clone()
    X_bp_test  = X_bp_t_htc[test_idx].clone()

    # 🔥 COVARIANCE ENABLED
    X_cov_train = X_cov_t_htc[train_idx].clone()
    X_cov_test  = X_cov_t_htc[test_idx].clone()

    X_plv_train = X_plv_t_htc[train_idx].clone()
    X_plv_test  = X_plv_t_htc[test_idx].clone()

    y_train = y_t_htc[train_idx].clone()
    y_test  = y_t_htc[test_idx].clone()

    subj_train = subjects_htc[train_idx]
    subj_test  = subjects_htc[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================

    # TIME
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # BANDPOWER
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # 🔥 COV NORMALIZATION (NEW)
    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    # ROI disabled (consistent with your setup)
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train,
        X_plv_train, X_roi_train, y_train, subj_train
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test,
        X_plv_test, X_roi_test, y_test, subj_test
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds, batch_size=BATCH, subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT (UNCHANGED)
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("HTC AdaBN + Mahalanobis few-shot acc:", fewshot_acc)

    overall_results_htc.append((test_subj, fewshot_acc))


print("\nHTC FINAL SUMMARY")
accs = [a for _, a in overall_results_htc]
for s, a in overall_results_htc:
    print(s, ":", a)

print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


HTC FOLD 1/17 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5505 | best 0.5505
Epoch 02 | zero-shot acc 0.4072 | best 0.5505
Epoch 03 | zero-shot acc 0.5668 | best 0.5668
Epoch 04 | zero-shot acc 0.6482 | best 0.6482
Epoch 05 | zero-shot acc 0.6189 | best 0.6482
Epoch 06 | zero-shot acc 0.6156 | best 0.6482
Epoch 07 | zero-shot acc 0.6417 | best 0.6482
Epoch 08 | zero-shot acc 0.6482 | best 0.6482
Epoch 09 | zero-shot acc 0.5570 | best 0.6482
Epoch 10 | zero-shot acc 0.5733 | best 0.6482
HTC AdaBN + Mahalanobis few-shot acc: 0.7265917658805847

HTC FOLD 2/17 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.5217 | best 0.5217
Epoch 02 | zero-shot acc 0.5518 | best 0.5518
Epoch 03 | zero-shot acc 0.6455 | best 0.6455
Epoch 04 | zero-shot acc 0.7157 | best 0.7157
Epoch 05 | zero-shot acc 0.6656 | best 0.7157
Epoch 06 | zero-shot acc 0.6321 | best 0.7157
Epoch 07 | zero-shot acc 0.5819 | best 0.7157
Epoch 08 | zero-shot acc 0.5819 | best 0.7157
Epoch 09 | zero-shot acc 0.58

In [ ]:
'''here i increased few shot to 30 per class from 20 per class'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10  # ↓ reduced to avoid overfitting
BATCH  = 396
LR     = 5e-4
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 30
SHRINK = 0.4

overall_results_htc = []

for fold_i, test_subj in enumerate(all_subjects_htc):

    print("\n" + "="*80)
    print(f"HTC FOLD {fold_i+1}/{len(all_subjects_htc)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects_htc == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_time_train = X_time_t_htc[train_idx].clone()
    X_time_test  = X_time_t_htc[test_idx].clone()

    X_bp_train = X_bp_t_htc[train_idx].clone()
    X_bp_test  = X_bp_t_htc[test_idx].clone()

    # 🔥 COVARIANCE ENABLED
    X_cov_train = X_cov_t_htc[train_idx].clone()
    X_cov_test  = X_cov_t_htc[test_idx].clone()

    X_plv_train = X_plv_t_htc[train_idx].clone()
    X_plv_test  = X_plv_t_htc[test_idx].clone()

    y_train = y_t_htc[train_idx].clone()
    y_test  = y_t_htc[test_idx].clone()

    subj_train = subjects_htc[train_idx]
    subj_test  = subjects_htc[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================

    # TIME
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # BANDPOWER
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # 🔥 COV NORMALIZATION (NEW)
    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    # ROI disabled (consistent with your setup)
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train,
        X_plv_train, X_roi_train, y_train, subj_train
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test,
        X_plv_test, X_roi_test, y_test, subj_test
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds, batch_size=BATCH, subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT (UNCHANGED)
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("HTC AdaBN + Mahalanobis few-shot acc:", fewshot_acc)

    overall_results_htc.append((test_subj, fewshot_acc))


print("\nHTC FINAL SUMMARY")
accs = [a for _, a in overall_results_htc]
for s, a in overall_results_htc:
    print(s, ":", a)

print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


HTC FOLD 1/17 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5668 | best 0.5668
Epoch 02 | zero-shot acc 0.5472 | best 0.5668
Epoch 03 | zero-shot acc 0.5863 | best 0.5863
Epoch 04 | zero-shot acc 0.6352 | best 0.6352
Epoch 05 | zero-shot acc 0.6417 | best 0.6417
Epoch 06 | zero-shot acc 0.6417 | best 0.6417
Epoch 07 | zero-shot acc 0.5733 | best 0.6417
Epoch 08 | zero-shot acc 0.5440 | best 0.6417
Epoch 09 | zero-shot acc 0.5570 | best 0.6417
Epoch 10 | zero-shot acc 0.5342 | best 0.6417
HTC AdaBN + Mahalanobis few-shot acc: 0.7287449836730957

HTC FOLD 2/17 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4615 | best 0.4615
Epoch 02 | zero-shot acc 0.4983 | best 0.4983
Epoch 03 | zero-shot acc 0.6923 | best 0.6923
Epoch 04 | zero-shot acc 0.6890 | best 0.6923
Epoch 05 | zero-shot acc 0.6221 | best 0.6923
Epoch 06 | zero-shot acc 0.5920 | best 0.6923
Epoch 07 | zero-shot acc 0.5786 | best 0.6923
Epoch 08 | zero-shot acc 0.5719 | best 0.6923
Epoch 09 | zero-shot acc 0.59

In [ ]:
''' here i kept few shot @ 20, and made shrink 0.7'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10  # ↓ reduced to avoid overfitting
BATCH  = 396
LR     = 5e-4
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.7

overall_results_htc = []

for fold_i, test_subj in enumerate(all_subjects_htc):

    print("\n" + "="*80)
    print(f"HTC FOLD {fold_i+1}/{len(all_subjects_htc)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects_htc == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_time_train = X_time_t_htc[train_idx].clone()
    X_time_test  = X_time_t_htc[test_idx].clone()

    X_bp_train = X_bp_t_htc[train_idx].clone()
    X_bp_test  = X_bp_t_htc[test_idx].clone()

    # 🔥 COVARIANCE ENABLED
    X_cov_train = X_cov_t_htc[train_idx].clone()
    X_cov_test  = X_cov_t_htc[test_idx].clone()

    X_plv_train = X_plv_t_htc[train_idx].clone()
    X_plv_test  = X_plv_t_htc[test_idx].clone()

    y_train = y_t_htc[train_idx].clone()
    y_test  = y_t_htc[test_idx].clone()

    subj_train = subjects_htc[train_idx]
    subj_test  = subjects_htc[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================

    # TIME
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # BANDPOWER
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # 🔥 COV NORMALIZATION (NEW)
    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    # ROI disabled (consistent with your setup)
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train,
        X_plv_train, X_roi_train, y_train, subj_train
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test,
        X_plv_test, X_roi_test, y_test, subj_test
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds, batch_size=BATCH, subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT (UNCHANGED)
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("HTC AdaBN + Mahalanobis few-shot acc:", fewshot_acc)

    overall_results_htc.append((test_subj, fewshot_acc))


print("\nHTC FINAL SUMMARY")
accs = [a for _, a in overall_results_htc]
for s, a in overall_results_htc:
    print(s, ":", a)

print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


HTC FOLD 1/17 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5147 | best 0.5147
Epoch 02 | zero-shot acc 0.4625 | best 0.5147
Epoch 03 | zero-shot acc 0.5603 | best 0.5603
Epoch 04 | zero-shot acc 0.6221 | best 0.6221
Epoch 05 | zero-shot acc 0.5765 | best 0.6221
Epoch 06 | zero-shot acc 0.5668 | best 0.6221
Epoch 07 | zero-shot acc 0.6254 | best 0.6254
Epoch 08 | zero-shot acc 0.5700 | best 0.6254
Epoch 09 | zero-shot acc 0.5375 | best 0.6254
Epoch 10 | zero-shot acc 0.5505 | best 0.6254
HTC AdaBN + Mahalanobis few-shot acc: 0.7677902579307556

HTC FOLD 2/17 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.5753 | best 0.5753
Epoch 02 | zero-shot acc 0.5719 | best 0.5753
Epoch 03 | zero-shot acc 0.7023 | best 0.7023
Epoch 04 | zero-shot acc 0.6923 | best 0.7023
Epoch 05 | zero-shot acc 0.6388 | best 0.7023
Epoch 06 | zero-shot acc 0.5318 | best 0.7023
Epoch 07 | zero-shot acc 0.5920 | best 0.7023
Epoch 08 | zero-shot acc 0.5452 | best 0.7023
Epoch 09 | zero-shot acc 0.50

In [ ]:
'''a do over of the same thing'''

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10  # ↓ reduced to avoid overfitting
BATCH  = 396
LR     = 5e-4
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.7

overall_results_htc = []

for fold_i, test_subj in enumerate(all_subjects_htc):

    print("\n" + "="*80)
    print(f"HTC FOLD {fold_i+1}/{len(all_subjects_htc)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects_htc == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_time_train = X_time_t_htc[train_idx].clone()
    X_time_test  = X_time_t_htc[test_idx].clone()

    X_bp_train = X_bp_t_htc[train_idx].clone()
    X_bp_test  = X_bp_t_htc[test_idx].clone()

    # 🔥 COVARIANCE ENABLED
    X_cov_train = X_cov_t_htc[train_idx].clone()
    X_cov_test  = X_cov_t_htc[test_idx].clone()

    X_plv_train = X_plv_t_htc[train_idx].clone()
    X_plv_test  = X_plv_t_htc[test_idx].clone()

    y_train = y_t_htc[train_idx].clone()
    y_test  = y_t_htc[test_idx].clone()

    subj_train = subjects_htc[train_idx]
    subj_test  = subjects_htc[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================

    # TIME
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # BANDPOWER
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # 🔥 COV NORMALIZATION (NEW)
    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    # ROI disabled (consistent with your setup)
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train,
        X_plv_train, X_roi_train, y_train, subj_train
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test,
        X_plv_test, X_roi_test, y_test, subj_test
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds, batch_size=BATCH, subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT (UNCHANGED)
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("HTC AdaBN + Mahalanobis few-shot acc:", fewshot_acc)

    overall_results_htc.append((test_subj, fewshot_acc))


print("\nHTC FINAL SUMMARY")
accs = [a for _, a in overall_results_htc]
for s, a in overall_results_htc:
    print(s, ":", a)

print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


HTC FOLD 1/17 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.5440 | best 0.5440
Epoch 02 | zero-shot acc 0.5668 | best 0.5668
Epoch 03 | zero-shot acc 0.5765 | best 0.5765
Epoch 04 | zero-shot acc 0.6612 | best 0.6612
Epoch 05 | zero-shot acc 0.6547 | best 0.6612
Epoch 06 | zero-shot acc 0.5961 | best 0.6612
Epoch 07 | zero-shot acc 0.6319 | best 0.6612
Epoch 08 | zero-shot acc 0.6189 | best 0.6612
Epoch 09 | zero-shot acc 0.6124 | best 0.6612
Epoch 10 | zero-shot acc 0.6580 | best 0.6612
HTC AdaBN + Mahalanobis few-shot acc: 0.7677902579307556

HTC FOLD 2/17 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.5184 | best 0.5184
Epoch 02 | zero-shot acc 0.4983 | best 0.5184
Epoch 03 | zero-shot acc 0.5920 | best 0.5920
Epoch 04 | zero-shot acc 0.6789 | best 0.6789
Epoch 05 | zero-shot acc 0.6254 | best 0.6789
Epoch 06 | zero-shot acc 0.5084 | best 0.6789
Epoch 07 | zero-shot acc 0.5084 | best 0.6789
Epoch 08 | zero-shot acc 0.5853 | best 0.6789
Epoch 09 | zero-shot acc 0.54

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10  # ↓ reduced to avoid overfitting
BATCH  = 396
LR     = 5e-4
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.7

overall_results_htc = []

for fold_i, test_subj in enumerate(all_subjects_htc):

    print("\n" + "="*80)
    print(f"HTC FOLD {fold_i+1}/{len(all_subjects_htc)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects_htc == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_time_train = X_time_t_htc[train_idx].clone()
    X_time_test  = X_time_t_htc[test_idx].clone()

    X_bp_train = X_bp_t_htc[train_idx].clone()
    X_bp_test  = X_bp_t_htc[test_idx].clone()

    # 🔥 COVARIANCE ENABLED
    X_cov_train = X_cov_t_htc[train_idx].clone()
    X_cov_test  = X_cov_t_htc[test_idx].clone()

    X_plv_train = X_plv_t_htc[train_idx].clone()
    X_plv_test  = X_plv_t_htc[test_idx].clone()

    y_train = y_t_htc[train_idx].clone()
    y_test  = y_t_htc[test_idx].clone()

    subj_train = subjects_htc[train_idx]
    subj_test  = subjects_htc[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================

    # TIME
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # BANDPOWER
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # 🔥 COV NORMALIZATION (NEW)
    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    # ROI disabled (consistent with your setup)
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train,
        X_plv_train, X_roi_train, y_train, subj_train
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test,
        X_plv_test, X_roi_test, y_test, subj_test
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds, batch_size=BATCH, subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)

    # =========================
    # FEW-SHOT (UNCHANGED)
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    all_e, all_y = extract_embeddings(model, test_loader_full, device)

    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - SHRINK) * cov + SHRINK * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)
    fewshot_acc = (pred == y_query).float().mean().item()

    print("HTC AdaBN + Mahalanobis few-shot acc:", fewshot_acc)

    overall_results_htc.append((test_subj, fewshot_acc))


print("\nHTC FINAL SUMMARY")
accs = [a for _, a in overall_results_htc]
for s, a in overall_results_htc:
    print(s, ":", a)

print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


HTC FOLD 1/17 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.4756 | best 0.4756
Epoch 02 | zero-shot acc 0.5798 | best 0.5798
Epoch 03 | zero-shot acc 0.6124 | best 0.6124
Epoch 04 | zero-shot acc 0.6189 | best 0.6189
Epoch 05 | zero-shot acc 0.6091 | best 0.6189
Epoch 06 | zero-shot acc 0.6091 | best 0.6189
Epoch 07 | zero-shot acc 0.6319 | best 0.6319
Epoch 08 | zero-shot acc 0.6384 | best 0.6384
Epoch 09 | zero-shot acc 0.6547 | best 0.6547
Epoch 10 | zero-shot acc 0.6287 | best 0.6547
HTC AdaBN + Mahalanobis few-shot acc: 0.7415730357170105

HTC FOLD 2/17 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.4916 | best 0.4916
Epoch 02 | zero-shot acc 0.5485 | best 0.5485
Epoch 03 | zero-shot acc 0.6689 | best 0.6689
Epoch 04 | zero-shot acc 0.6722 | best 0.6722
Epoch 05 | zero-shot acc 0.6823 | best 0.6823
Epoch 06 | zero-shot acc 0.6254 | best 0.6823
Epoch 07 | zero-shot acc 0.5819 | best 0.6823
Epoch 08 | zero-shot acc 0.5987 | best 0.6823
Epoch 09 | zero-shot acc 0.52

In [ ]:
'''subject test time adapatation '''

In [ ]:
'''added:

cov gate ✅

plv gate ✅

shrink tuning ✅

…and got ~same or slightly worse performance
'''

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FusionEncoder(nn.Module):
    def __init__(self, emb_dim=64, bp_dim=42, roi_dim=3):
        super().__init__()

        total_bp_dim = bp_dim + roi_dim

        # ------------------
        # TIME BRANCH
        # ------------------
        self.time_branch = nn.Sequential(
            nn.Conv1d(14, 14, kernel_size=7, padding=3, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 14, kernel_size=15, padding=7, groups=14),
            nn.BatchNorm1d(14),
            nn.GELU(),

            nn.Conv1d(14, 32, kernel_size=1),
            nn.BatchNorm1d(32),
            nn.GELU(),
        )

        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(32, 8, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(8, 32, kernel_size=1),
            nn.Sigmoid()
        )

        self.attn_pool = nn.Conv1d(32, 1, kernel_size=1)

        # ------------------
        # BANDPOWER + ROI BRANCH
        # ------------------
        self.bp_branch = nn.Sequential(
            nn.Linear(total_bp_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )

        # ------------------
        # COVARIANCE BRANCH
        # ------------------
        self.cov_branch = nn.Sequential(
            nn.Linear(105, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # PLV BRANCH
        # ------------------
        self.plv_branch = nn.Sequential(
            nn.Linear(210, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32)
        )

        # ------------------
        # FUSION
        # ------------------
        self.embed = nn.Sequential(
            nn.Linear(32 + 32 + 32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(128, emb_dim)
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi,
                cov_gate=1.0, plv_gate=1.0):

        # ------------------
        # Time branch
        # ------------------
        x = self.time_branch(x_time)
        scale = self.channel_attn(x)
        x = x * scale

        weights = self.attn_pool(x)
        weights = F.softmax(weights, dim=-1)
        z_time = (x * weights).sum(dim=-1)   # (B, 32)

        # ------------------
        # Bandpower + ROI
        # ------------------
        bp_in = torch.cat([x_bp, x_roi], dim=1)
        z_bp = self.bp_branch(bp_in)

        # ------------------
        # Covariance
        # ------------------
        z_cov = self.cov_branch(x_cov)

        # ------------------
        # PLV
        # ------------------
        z_plv = self.plv_branch(x_plv)

        # ------------------
        # Test-time modality gates
        # ------------------
        if not torch.is_tensor(cov_gate):
            cov_gate = torch.tensor(cov_gate, device=z_cov.device, dtype=z_cov.dtype)
        if not torch.is_tensor(plv_gate):
            plv_gate = torch.tensor(plv_gate, device=z_plv.device, dtype=z_plv.dtype)

        z_cov = z_cov * cov_gate
        z_plv = z_plv * plv_gate

        # ------------------
        # Fusion
        # ------------------
        z = torch.cat([z_time, z_bp, z_cov, z_plv], dim=1)
        e = self.embed(z)

        return e


class ContrastiveModel(nn.Module):
    def __init__(self, emb_dim=64, proj_dim=64, bp_dim=42, roi_dim=3):
        super().__init__()

        self.encoder = FusionEncoder(
            emb_dim=emb_dim,
            bp_dim=bp_dim,
            roi_dim=roi_dim
        )

        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi,
                cov_gate=1.0, plv_gate=1.0):
        e = self.encoder(
            x_time, x_bp, x_cov, x_plv, x_roi,
            cov_gate=cov_gate,
            plv_gate=plv_gate
        )
        p = self.proj(e)
        return e, p

In [ ]:
def extract_embeddings(model, loader, device, cov_gate=1.0, plv_gate=1.0):
    model.eval()
    E = []
    Y = []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in loader:
            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            e, _ = model(
                xb_time, xb_bp, xb_cov, xb_plv, xb_roi,
                cov_gate=cov_gate,
                plv_gate=plv_gate
            )

            E.append(e.cpu())
            Y.append(yb.cpu())

    E = torch.cat(E, dim=0)
    Y = torch.cat(Y, dim=0)
    return E, Y

In [ ]:
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

EPOCHS = 10  # ↓ reduced to avoid overfitting
BATCH  = 396
LR     = 5e-4
LAMBDA = 0.5
TEMP   = 0.1
PROTO_LAMBDA = 0.10

FEW_SHOT_PER_CLASS = 20
SHRINK = 0.7

overall_results_htc = []

for fold_i, test_subj in enumerate(all_subjects_htc):

    print("\n" + "="*80)
    print(f"HTC FOLD {fold_i+1}/{len(all_subjects_htc)} | TEST SUBJECT = {test_subj}")
    print("="*80)

    # =========================
    # SPLIT
    # =========================
    test_mask  = (subjects_htc == test_subj)
    train_mask = ~test_mask

    train_idx = np.where(train_mask)[0]
    test_idx  = np.where(test_mask)[0]

    # =========================
    # FEATURE PREP
    # =========================
    X_time_train = X_time_t_htc[train_idx].clone()
    X_time_test  = X_time_t_htc[test_idx].clone()

    X_bp_train = X_bp_t_htc[train_idx].clone()
    X_bp_test  = X_bp_t_htc[test_idx].clone()

    # 🔥 COVARIANCE ENABLED
    X_cov_train = X_cov_t_htc[train_idx].clone()
    X_cov_test  = X_cov_t_htc[test_idx].clone()

    X_plv_train = X_plv_t_htc[train_idx].clone()
    X_plv_test  = X_plv_t_htc[test_idx].clone()

    y_train = y_t_htc[train_idx].clone()
    y_test  = y_t_htc[test_idx].clone()

    subj_train = subjects_htc[train_idx]
    subj_test  = subjects_htc[test_idx]

    # =========================
    # NORMALIZATION (train-only)
    # =========================

    # TIME
    time_mu = X_time_train.mean(dim=(0,2), keepdim=True)
    time_sd = X_time_train.std(dim=(0,2), keepdim=True) + 1e-6
    X_time_train = (X_time_train - time_mu) / time_sd
    X_time_test  = (X_time_test  - time_mu) / time_sd

    # BANDPOWER
    bp_mu = X_bp_train.mean(dim=0, keepdim=True)
    bp_sd = X_bp_train.std(dim=0, keepdim=True) + 1e-6
    X_bp_train = (X_bp_train - bp_mu) / bp_sd
    X_bp_test  = (X_bp_test  - bp_mu) / bp_sd

    # 🔥 COV NORMALIZATION (NEW)
    cov_mu = X_cov_train.mean(dim=0, keepdim=True)
    cov_sd = X_cov_train.std(dim=0, keepdim=True) + 1e-6
    X_cov_train = (X_cov_train - cov_mu) / cov_sd
    X_cov_test  = (X_cov_test  - cov_mu) / cov_sd

    # ROI disabled (consistent with your setup)
    X_roi_train = torch.zeros((X_bp_train.shape[0], 0))
    X_roi_test  = torch.zeros((X_bp_test.shape[0], 0))

    # =========================
    # DATASET / LOADER
    # =========================
    train_ds = EEGDataset(
        X_time_train, X_bp_train, X_cov_train,
        X_plv_train, X_roi_train, y_train, subj_train
    )

    test_ds = EEGDataset(
        X_time_test, X_bp_test, X_cov_test,
        X_plv_test, X_roi_test, y_test, subj_test
    )

    sampler = SubjectBalancedBatchSampler(
        train_ds, batch_size=BATCH, subjects_per_batch=12
    )

    train_loader = DataLoader(train_ds, batch_sampler=sampler)
    test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

    # =========================
    # MODEL
    # =========================
    model = ContrastiveModel(
        emb_dim=64,
        proj_dim=64,
        bp_dim=42,
        roi_dim=0
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=LR)

    best_acc = -1.0
    best_state = None

    # =========================
    # TRAIN
    # =========================
    for ep in range(1, EPOCHS+1):
        model.train()

        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in train_loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)
            yb      = yb.to(device)
            sb      = sb.to(device)

            e, proj = model(xb_time, xb_bp, xb_cov, xb_plv, xb_roi)

            loss_metric = prototype_ce_loss(e, yb, temperature=0.1)
            loss_con    = supcon_diff_subject_loss(proj, yb, sb, temperature=TEMP)
            loss_proto  = prototype_loss(e, yb)

            loss = loss_metric + LAMBDA * loss_con + PROTO_LAMBDA * loss_proto

            opt.zero_grad()
            loss.backward()
            opt.step()

        # =========================
        # ZERO-SHOT EVAL
        # =========================
        model.eval()
        with torch.no_grad():

            train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)
            e_train, y_train_full = extract_embeddings(model, train_eval_loader, device)
            e_test,  y_test_full  = extract_embeddings(model, test_loader, device)

            classes = torch.unique(y_train_full)
            protos = torch.stack([
                e_train[y_train_full == c].mean(dim=0)
                for c in classes
            ])

            logits = F.normalize(e_test, dim=1) @ F.normalize(protos, dim=1).T
            pred_local = logits.argmax(dim=1)

            label_map = {int(c.item()): i for i, c in enumerate(classes)}
            y_test_local = torch.tensor(
                [label_map[int(v.item())] for v in y_test_full],
                device=pred_local.device
            )

            acc = (pred_local == y_test_local).float().mean().item()

        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {ep:02d} | zero-shot acc {acc:.4f} | best {best_acc:.4f}")

    model.load_state_dict(best_state)



    # =========================
    # FEW-SHOT ADAPTIVE
    # =========================
    torch.manual_seed(1234 + fold_i)
    np.random.seed(1234 + fold_i)

    test_loader_full = DataLoader(test_ds, batch_size=256, shuffle=False)

    # initial embeddings with default gates
    all_e_init, all_y = extract_embeddings(
        model, test_loader_full, device,
        cov_gate=1.0, plv_gate=1.0
    )

    # stratified support selection
    support_idx = []
    for c in torch.unique(all_y):
        class_idx = (all_y == c).nonzero(as_tuple=True)[0]
        perm = torch.randperm(len(class_idx))
        support_idx.append(class_idx[perm[:FEW_SHOT_PER_CLASS]])
    support_idx = torch.cat(support_idx)

    mask = torch.zeros(len(all_y), dtype=torch.bool)
    mask[support_idx] = True

    # support_subset = torch.utils.data.Subset(test_ds, support_idx.tolist())
    support_subset = torch.utils.data.Subset(
    test_ds,
    support_idx.cpu().numpy().tolist()
    )
    support_loader = DataLoader(support_subset, batch_size=256, shuffle=False)

    # --- tiny support-only tuning grid ---
    cov_gate_grid = [0.0, 0.25, 0.5, 0.75, 1.0]
    plv_gate_grid = [0.5, 1.0]
    shrink_grid   = [0.5, 0.7, 0.9]

    best_cfg = None
    best_support_score = -1.0

    # Use support-set prototype compactness/separation proxy
    for cov_gate in cov_gate_grid:
        for plv_gate in plv_gate_grid:
            e_sup_all, y_sup_all = extract_embeddings(
                model, support_loader, device,
                cov_gate=cov_gate,
                plv_gate=plv_gate
            )

            e_sup_all = e_sup_all.to(device)
            y_sup_all = y_sup_all.to(device)

            # embedding weighting (same as before)
            e_sup_all = F.normalize(e_sup_all, dim=1)
            var = e_sup_all.var(dim=0)
            weights = 1.0 / (var + 1e-6)
            weights = weights / weights.mean()
            e_sup_w = e_sup_all * weights

            classes = torch.unique(y_sup_all)
            protos = torch.stack([e_sup_w[y_sup_all == c].mean(dim=0) for c in classes])
            protos = F.normalize(protos, dim=1)

            # support compactness/separation proxy
            within = 0.0
            for i, c in enumerate(classes):
                class_emb = e_sup_w[y_sup_all == c]
                within += ((class_emb - protos[i]) ** 2).sum(dim=1).mean()
            within = within / len(classes)

            if len(classes) >= 2:
                between = ((protos[0] - protos[1]) ** 2).sum()
            else:
                between = torch.tensor(0.0, device=device)

            support_score = (between / (within + 1e-6)).item()

            # prefer configs with stronger support separation
            if support_score > best_support_score:
                best_support_score = support_score
                best_cfg = {
                    "cov_gate": cov_gate,
                    "plv_gate": plv_gate,
                    "weights": weights.detach().clone(),
                }

    # use best gates
    best_cov_gate = best_cfg["cov_gate"]
    best_plv_gate = best_cfg["plv_gate"]

    all_e, all_y = extract_embeddings(
        model, test_loader_full, device,
        cov_gate=best_cov_gate,
        plv_gate=best_plv_gate
    )

    e_support = all_e[mask].to(device)
    y_support = all_y[mask].to(device)
    e_query   = all_e[~mask].to(device)
    y_query   = all_y[~mask].to(device)

    e_support = F.normalize(e_support, dim=1)
    e_query   = F.normalize(e_query, dim=1)

    # support-derived embedding weights
    var = e_support.var(dim=0)
    weights = 1.0 / (var + 1e-6)
    weights = weights / weights.mean()

    e_support = e_support * weights
    e_query   = e_query * weights

    classes = torch.unique(y_support)
    protos = torch.stack([e_support[y_support == c].mean(dim=0) for c in classes])
    protos = F.normalize(protos, dim=1)

    residuals = torch.cat([
        e_support[y_support == c] - protos[i]
        for i, c in enumerate(classes)
    ], dim=0)

    D = residuals.shape[1]
    cov = (residuals.t() @ residuals) / max(residuals.shape[0] - 1, 1)

    # adaptive shrink: choose from small grid using support compactness proxy
    best_shrink = None
    best_shrink_score = -1e9

    for shrink in shrink_grid:
        avg_var = torch.trace(cov) / D
        cov_shrink = (1 - shrink) * cov + shrink * (avg_var * torch.eye(D, device=device))
        cov_inv = torch.linalg.pinv(cov_shrink)

        diffs_sup = e_support.unsqueeze(1) - protos.unsqueeze(0)
        dists_sup = torch.einsum("nkd,dd,nkd->nk", diffs_sup, cov_inv, diffs_sup)

        pred_sup = dists_sup.argmin(dim=1)

        label_map = {int(c.item()): i for i, c in enumerate(classes)}
        y_support_local = torch.tensor(
            [label_map[int(v.item())] for v in y_support],
            device=device
        )

        sup_acc = (pred_sup == y_support_local).float().mean().item()

        if sup_acc > best_shrink_score:
            best_shrink_score = sup_acc
            best_shrink = shrink

    avg_var = torch.trace(cov) / D
    cov_shrink = (1 - best_shrink) * cov + best_shrink * (avg_var * torch.eye(D, device=device))
    cov_inv = torch.linalg.pinv(cov_shrink)

    diffs = e_query.unsqueeze(1) - protos.unsqueeze(0)
    dists = torch.einsum("nkd,dd,nkd->nk", diffs, cov_inv, diffs)

    pred = dists.argmin(dim=1)

    label_map = {int(c.item()): i for i, c in enumerate(classes)}
    y_query_local = torch.tensor(
        [label_map[int(v.item())] for v in y_query],
        device=device
    )

    fewshot_acc = (pred == y_query_local).float().mean().item()

    print(f"Best cov_gate={best_cov_gate}, best plv_gate={best_plv_gate}, best shrink={best_shrink}")
    print("HTC AdaBN + adaptive-gate + Mahalanobis few-shot acc:", fewshot_acc)

    overall_results_htc.append((test_subj, fewshot_acc))

print("\nHTC FINAL SUMMARY")
accs = [a for _, a in overall_results_htc]
for s, a in overall_results_htc:
    print(s, ":", a)

print("Mean acc:", float(np.mean(accs)), "Std:", float(np.std(accs)))


HTC FOLD 1/17 | TEST SUBJECT = subject_01
Epoch 01 | zero-shot acc 0.3322 | best 0.3322
Epoch 02 | zero-shot acc 0.3322 | best 0.3322
Epoch 03 | zero-shot acc 0.3322 | best 0.3322
Epoch 04 | zero-shot acc 0.3322 | best 0.3322
Epoch 05 | zero-shot acc 0.3322 | best 0.3322
Epoch 06 | zero-shot acc 0.3322 | best 0.3322
Epoch 07 | zero-shot acc 0.3322 | best 0.3322
Epoch 08 | zero-shot acc 0.3322 | best 0.3322
Epoch 09 | zero-shot acc 0.3322 | best 0.3322
Epoch 10 | zero-shot acc 0.3322 | best 0.3322
Best cov_gate=0.0, best plv_gate=0.5, best shrink=0.5
HTC AdaBN + adaptive-gate + Mahalanobis few-shot acc: 0.7940074801445007

HTC FOLD 2/17 | TEST SUBJECT = subject_02
Epoch 01 | zero-shot acc 0.6254 | best 0.6254
Epoch 02 | zero-shot acc 0.6254 | best 0.6254
Epoch 03 | zero-shot acc 0.6254 | best 0.6254
Epoch 04 | zero-shot acc 0.6254 | best 0.6254
Epoch 05 | zero-shot acc 0.6254 | best 0.6254
Epoch 06 | zero-shot acc 0.6254 | best 0.6254
Epoch 07 | zero-shot acc 0.6254 | best 0.6254
Epoch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ContrastiveModel(nn.Module):
    def __init__(self,
                 emb_dim=64,
                 proj_dim=64,
                 bp_dim=42,
                 roi_dim=0,
                 num_classes=2,
                 num_datasets=2):

        super().__init__()

        self.encoder = FusionEncoder(
            emb_dim=emb_dim,
            bp_dim=bp_dim,
            roi_dim=roi_dim
        )

        # dataset embedding
        self.dataset_embed = nn.Embedding(num_datasets, 8)

        self.fusion_adjust = nn.Linear(emb_dim + 8, emb_dim)

        self.proj = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, proj_dim),
        )

        self.cls = nn.Linear(emb_dim, num_classes)

    def forward(self, x_time, x_bp, x_cov, x_plv, x_roi, dataset_id):

        e = self.encoder(x_time, x_bp, x_cov, x_plv, x_roi)

        d = self.dataset_embed(dataset_id)
        e = torch.cat([e, d], dim=1)
        e = self.fusion_adjust(e)

        p = self.proj(e)
        logits = self.cls(e)

        return e, p, logits

In [ ]:
def extract_embeddings(model, loader, device, dataset_id_val=0):
    model.eval()
    E, Y = [], []

    with torch.no_grad():
        for xb_time, xb_bp, xb_cov, xb_plv, xb_roi, yb, sb in loader:

            xb_time = xb_time.to(device)
            xb_bp   = xb_bp.to(device)
            xb_cov  = xb_cov.to(device)
            xb_plv  = xb_plv.to(device)
            xb_roi  = xb_roi.to(device)

            dataset_id = torch.full_like(yb, dataset_id_val).to(device)

            e, _, _ = model(
                xb_time, xb_bp, xb_cov, xb_plv, xb_roi, dataset_id
            )

            E.append(e.cpu())
            Y.append(yb.cpu())

    return torch.cat(E), torch.cat(Y)